# DeepVoice DACON — Accuracy·교차 데이터셋 강건성 중심 Colab 통합 워크플로우

ASVspoof 2019 LA/PA에서 시작해 Log-Mel/LFCC baseline → AASIST → RawBoost·실제 codec/RIR → XLS-R/WavLM/TCM → Mamba → 음성/음악 멀티브랜치 → 교차 데이터셋 강건성 평가 → 검증 기반 ensemble/calibration → DACON 코드 제출 패키지까지 연결하는 통합 노트북입니다.

핵심 목표는 한 데이터셋의 accuracy 최고값이 아니라 **합성기·언어·코덱·통신환경·replay·음악 domain이 바뀌어도 높은 accuracy를 유지하는 모델**입니다. Checkpoint와 조기 종료의 1차 기준은 accuracy이며, 클래스 불균형을 감시하기 위해 balanced accuracy·F1·EER·AUC도 함께 기록합니다. `SELECTED_EXPERIMENT`로 한 실험씩 실행하고 결과·체크포인트·도메인별 예측을 Google Drive에 누적합니다.

핵심 라벨 규약(ASVspoof): `0 = spoof/fake`, `1 = bonafide/real`. 모든 내부 score는 `P(bonafide)`로 저장합니다.


## 먼저 읽기 — 2026-08-20 기준 대회 상태와 규칙

- 대회: [딥보이스 범죄 대응을 위한 AI 탐지 모델 경진대회](https://dacon.io/competitions/official/236749/overview/description)
- 일정: 2026-08-26 시작, 2026-09-29 리더보드 제출 마감, 2026-09-30 종료.
- 오늘 기준 세부 데이터 설명, 평가 산식, 서버 사양, ZIP 용량·설치·추론 시간 제한은 **8월 26일 10:00 공개 예정**입니다.
- 제출 형식은 `submit.zip/{model/, script.py, requirements.txt}`이며 평가 서버는 인터넷이 차단됩니다.
- 비공개 TEST는 **추론 전용**입니다. 추가 학습, 튜닝, pseudo-labeling, TEST 전체 통계를 이용한 보정은 금지됩니다.
- 각 TEST 파일은 독립적으로 예측해야 합니다. 한 파일 내부 segment TTA는 허용되지만 다른 TEST 파일의 정보·예측·통계를 이용하면 안 됩니다.
- ASVspoof 등 외부 데이터와 공개 사전학습 모델은 현재 규칙상 사용할 수 있으나, 2차 평가 자료에 출처와 학습 데이터 전체를 명시·제출해야 합니다.

따라서 마지막 DACON 어댑터는 공개된 `sample_submission.csv`를 읽어 열을 검사하며, 알 수 없는 다중 출력 의미를 임의로 가정하지 않습니다. 공개일에 `DACON_SCHEMA`와 평가 산식만 확정하면 앞 단계의 학습 코드를 그대로 재사용할 수 있습니다.


## 0. Colab 런타임

권장: GPU 런타임. AASIST는 T4에서 batch 8, XLS-R 300M 계열은 batch 1–2와 gradient accumulation을 권장합니다. 아래 설치 셀 실행 후 import 오류가 남으면 런타임을 한 번 재시작합니다.


In [ ]:
%pip install -q "transformers>=4.48,<5" "accelerate>=1.2" "optuna>=4.0" \
  "soundfile>=0.12" "librosa>=0.10" "scikit-learn>=1.4" \
  "scipy>=1.11" "seaborn>=0.13" "pandas>=2.0"


In [ ]:
from __future__ import annotations

import gc
import hashlib
import importlib
import json
import math
import os
import random
import shutil
import subprocess
import sys
import tarfile
import time
import warnings
import zipfile
from dataclasses import asdict, dataclass
from pathlib import Path
from types import SimpleNamespace
from typing import Iterable, Sequence

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import soundfile as sf
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
from scipy.optimize import minimize
from sklearn.calibration import calibration_curve
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score, log_loss, roc_auc_score, roc_curve,
)
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from tqdm.auto import tqdm

warnings.filterwarnings("ignore", category=FutureWarning)

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")


@dataclass
class ProjectConfig:
    drive_root: str = "/content/drive/MyDrive/deepvoice_dacon"
    work_root: str = "/content/deepvoice_work"
    # ZIP 이름은 Drive에 올린 실제 파일명으로 수정합니다.
    asv_archives: tuple[str, ...] = (
        "ASVspoof2019_LA.zip",
        "ASVspoof2019_PA.zip",
    )
    dacon_archive: str = "dacon_data.zip"
    sample_rate: int = 16_000
    clip_samples: int = 64_600
    seed: int = 42
    num_workers: int = 0  # Drive I/O 문제를 피하는 안전한 시작값


CFG = ProjectConfig()
DRIVE_ROOT = Path(CFG.drive_root)
WORK_ROOT = Path(CFG.work_root)
DATA_ROOT = WORK_ROOT / "data"
REPO_ROOT = WORK_ROOT / "repos"
RUN_ROOT = DRIVE_ROOT / "runs_accuracy"
EXPORT_ROOT = DRIVE_ROOT / "exports_accuracy"
for directory in (DRIVE_ROOT, WORK_ROOT, DATA_ROOT, REPO_ROOT, RUN_ROOT, EXPORT_ROOT):
    directory.mkdir(parents=True, exist_ok=True)


def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


seed_everything(CFG.seed)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
    print("torch/cuda:", torch.__version__, torch.version.cuda)


## 1. Google Drive ZIP 해제

Drive에서 매 epoch마다 오디오를 읽으면 느립니다. ZIP 원본과 체크포인트는 Drive에 보존하고, 학습 데이터는 Colab 로컬 `/content/deepvoice_work/data`로 한 번만 풉니다. 압축 해제 완료 marker가 있으면 재실행 시 건너뜁니다.


In [ ]:
def extract_archive(archive: Path, destination: Path, force: bool = False) -> Path:
    if not archive.exists():
        print(f"[skip] archive not found: {archive}")
        return destination
    digest = hashlib.sha1(str(archive.resolve()).encode()).hexdigest()[:10]
    marker = destination / f".extracted_{archive.stem}_{digest}"
    if marker.exists() and not force:
        print(f"[cached] {archive.name}")
        return destination

    destination.mkdir(parents=True, exist_ok=True)
    print(f"[extract] {archive} -> {destination}")
    if zipfile.is_zipfile(archive):
        with zipfile.ZipFile(archive) as zf:
            zf.extractall(destination)
    elif tarfile.is_tarfile(archive):
        with tarfile.open(archive) as tf:
            tf.extractall(destination)
    else:
        raise ValueError(f"지원하지 않는 압축 형식: {archive}")
    marker.write_text("ok\n", encoding="utf-8")
    return destination


EXTRACT_ASVSPOOF = True
EXTRACT_DACON = True  # 공개 전 파일이 없으면 안전하게 skip

if EXTRACT_ASVSPOOF:
    for name in CFG.asv_archives:
        extract_archive(DRIVE_ROOT / name, DATA_ROOT / "asvspoof2019")
if EXTRACT_DACON:
    extract_archive(DRIVE_ROOT / CFG.dacon_archive, DATA_ROOT / "dacon")


## 2. 공식 구현 준비 및 출처 고정

- AASIST: `clovaai/aasist` (MIT)
- RawBoost: `TakHemlata/RawBoost-antispoofing`
- SSL anti-spoofing: `TakHemlata/SSL_Anti-spoofing` (MIT, 선택 실험)
- XLSR-Mamba: `swagshaw/XLSR-Mamba` (MIT, 선택 실험)

핵심 노트북은 현재 Colab의 PyTorch를 유지합니다. 오래된 fairseq 버전을 요구하는 SSL/Mamba 공식 재현은 별도 런타임에서 실행하는 것이 안전하므로 기본 clone 대상에서 제외했습니다. 사용한 저장소 commit은 자동 기록됩니다.


In [ ]:
REPOSITORIES = {
    "aasist": "https://github.com/clovaai/aasist.git",
    "rawboost": "https://github.com/TakHemlata/RawBoost-antispoofing.git",
    "ssl_antispoof": "https://github.com/TakHemlata/SSL_Anti-spoofing.git",
    "xlsr_mamba": "https://github.com/swagshaw/XLSR-Mamba.git",
}


def clone_repo(name: str, optional: bool = False) -> Path:
    destination = REPO_ROOT / name
    if not destination.exists():
        print("cloning", REPOSITORIES[name])
        subprocess.run(
            ["git", "clone", "--depth", "1", REPOSITORIES[name], str(destination)],
            check=True,
        )
    commit = subprocess.check_output(
        ["git", "-C", str(destination), "rev-parse", "HEAD"], text=True
    ).strip()
    print(f"{name}: {commit}")
    return destination


CLONE_CORE_REPOS = True
CLONE_LEGACY_ADVANCED_REPOS = False

REPO_COMMITS = {}
if CLONE_CORE_REPOS:
    for repo_name in ("aasist", "rawboost"):
        repo_path = clone_repo(repo_name)
        REPO_COMMITS[repo_name] = subprocess.check_output(
            ["git", "-C", str(repo_path), "rev-parse", "HEAD"], text=True
        ).strip()
if CLONE_LEGACY_ADVANCED_REPOS:
    for repo_name in ("ssl_antispoof", "xlsr_mamba"):
        repo_path = clone_repo(repo_name, optional=True)
        REPO_COMMITS[repo_name] = subprocess.check_output(
            ["git", "-C", str(repo_path), "rev-parse", "HEAD"], text=True
        ).strip()


## 3. ASVspoof 2019 LA/PA 인덱스 생성

공식 protocol의 train/dev/eval split을 그대로 사용합니다. Random split을 다시 만들지 않습니다. LA와 PA를 합칠 때는 `track × label` 균형 sampler를 사용해 PA 또는 spoof 다수가 학습을 지배하지 않게 합니다.


In [ ]:
AUDIO_SUFFIXES = {".wav", ".flac", ".ogg", ".mp3", ".m4a", ".aac"}


def find_protocol(root: Path, track: str, split: str) -> Path | None:
    candidates = []
    for path in root.rglob("*.txt"):
        name = path.name.lower()
        full = str(path).lower()
        if track.lower() in full and split.lower() in name and "protocol" in full:
            candidates.append(path)
    if not candidates:
        return None
    candidates.sort(key=lambda p: ("cm." not in p.name.lower(), len(str(p))))
    return candidates[0]


def build_audio_map(root: Path) -> dict[str, Path]:
    mapping = {}
    for path in tqdm(root.rglob("*"), desc="Audio index"):
        if path.is_file() and path.suffix.lower() in AUDIO_SUFFIXES:
            mapping.setdefault(path.stem, path)
    return mapping


def parse_asvspoof_protocol(protocol: Path, audio_map: dict[str, Path], track: str, split: str) -> pd.DataFrame:
    rows = []
    for line_no, line in enumerate(protocol.read_text(encoding="utf-8").splitlines(), 1):
        parts = line.split()
        if len(parts) < 5:
            raise ValueError(f"잘못된 protocol line {protocol}:{line_no}: {line}")
        speaker, utt_id, environment, attack, key = parts[:5]
        key = key.lower()
        if key not in {"bonafide", "spoof"}:
            raise ValueError(f"알 수 없는 label {key}: {protocol}:{line_no}")
        rows.append({
            "id": utt_id,
            "path": str(audio_map.get(utt_id, "")),
            "label": 1 if key == "bonafide" else 0,
            "key": key,
            "speaker": speaker,
            "environment": environment,
            "attack": attack,
            "track": track,
            "split": split,
            "source": "ASVspoof2019",
        })
    frame = pd.DataFrame(rows)
    missing = frame[frame["path"] == ""]
    if len(missing):
        examples = missing["id"].head().tolist()
        raise FileNotFoundError(f"protocol 오디오 {len(missing)}개를 찾지 못함. 예: {examples}")
    return frame


def build_asvspoof_index(root: Path) -> pd.DataFrame:
    if not root.exists():
        print("ASVspoof root가 없습니다:", root)
        return pd.DataFrame()
    audio_map = build_audio_map(root)
    frames = []
    for track in ("LA", "PA"):
        for split in ("train", "dev", "eval"):
            protocol = find_protocol(root, track, split)
            if protocol is None:
                print(f"[warn] protocol 없음: {track}/{split}")
                continue
            print(f"{track}/{split}: {protocol}")
            frames.append(parse_asvspoof_protocol(protocol, audio_map, track, split))
    if not frames:
        return pd.DataFrame()
    frame = pd.concat(frames, ignore_index=True)
    assert not frame.duplicated(["track", "split", "id"]).any()
    return frame


ASV_ROOT = DATA_ROOT / "asvspoof2019"
asv_df = build_asvspoof_index(ASV_ROOT)
if len(asv_df):
    display(pd.crosstab([asv_df["track"], asv_df["split"]], asv_df["key"], margins=True))
    display(asv_df.head())


## 4. DACON 공개 데이터 계약 검사

8월 26일 이후 실행합니다. 파일명·열 이름을 추측해 학습하는 대신 실제 CSV와 오디오를 먼저 출력하고 검증합니다. `test`에는 label이 없어야 하며, 이후 학습 파이프라인에는 오직 공개 train과 외부 학습 데이터만 넣습니다.


In [ ]:
DACON_ROOT = DATA_ROOT / "dacon"


def csv_inventory(root: Path) -> dict[str, Path]:
    if not root.exists():
        return {}
    return {p.name.lower(): p for p in root.rglob("*.csv")}


def audio_inventory(root: Path) -> pd.DataFrame:
    if not root.exists():
        return pd.DataFrame(columns=["id", "path"])
    paths = [p for p in root.rglob("*") if p.is_file() and p.suffix.lower() in AUDIO_SUFFIXES]
    return pd.DataFrame({"id": [p.stem for p in paths], "path": [str(p) for p in paths]})


def inspect_dacon_contract(root: Path) -> dict:
    csvs = csv_inventory(root)
    print("CSV files:", {k: str(v) for k, v in csvs.items()})
    result = {"csvs": csvs, "train": None, "test": None, "sample": None}
    for key, path in csvs.items():
        frame = pd.read_csv(path)
        print(f"\n[{path.name}] shape={frame.shape}")
        display(frame.head())
        print(frame.dtypes)
        if "sample" in key and "submission" in key:
            result["sample"] = frame
            result["sample_path"] = path
        elif "train" in key:
            result["train"] = frame
            result["train_path"] = path
        elif "test" in key:
            result["test"] = frame
            result["test_path"] = path
    audio_df = audio_inventory(root)
    print("audio files:", len(audio_df), "suffixes:", sorted({Path(x).suffix for x in audio_df["path"]}))
    result["audio"] = audio_df
    return result


dacon_contract = inspect_dacon_contract(DACON_ROOT)

# 공개 후 반드시 명시적으로 확인/수정할 값. None이면 sample_submission에서 보수적으로 추론합니다.
DACON_SCHEMA = {
    "id_col": None,
    "audio_col": None,
    "target_cols": None,
    "group_col": None,  # speaker/source/generator 열이 있으면 지정
    # 단일 이진 출력일 때만 사용: "spoof" 또는 "bonafide"
    "single_target_positive": "spoof",
}


def infer_schema(contract: dict, overrides: dict) -> dict:
    sample = contract.get("sample")
    test = contract.get("test")
    if sample is None:
        raise FileNotFoundError("sample_submission.csv가 없습니다. 8월 26일 공개 데이터를 확인하세요.")
    id_col = overrides.get("id_col") or sample.columns[0]
    target_cols = overrides.get("target_cols") or [c for c in sample.columns if c != id_col]
    audio_col = overrides.get("audio_col")
    if audio_col is None and test is not None:
        candidates = [c for c in test.columns if any(k in c.lower() for k in ("path", "file", "audio", "name"))]
        audio_col = candidates[0] if candidates else None
    return {**overrides, "id_col": id_col, "target_cols": target_cols, "audio_col": audio_col}


def prepare_dacon_supervised_frames(contract: dict, schema: dict, dev_size: float = 0.2):
    """공개 train만 사용해 train/dev를 만든다. TEST는 이 함수 입력에 들어오지 않는다."""
    train = contract.get("train")
    if train is None:
        raise FileNotFoundError("공개 train.csv가 없습니다.")
    train = train.copy()
    id_col = schema["id_col"]
    target_cols = list(schema["target_cols"])
    missing_targets = [c for c in target_cols if c not in train.columns]
    if missing_targets:
        raise KeyError(f"train.csv에 target 열이 없습니다: {missing_targets}")

    audio = contract["audio"].copy()
    audio["audio_key"] = audio["id"].astype(str).map(lambda x: Path(x).stem)
    if audio["audio_key"].duplicated().any():
        raise ValueError("오디오 stem이 중복됩니다. train/test 하위 폴더를 포함하도록 path resolver를 수정하세요.")
    audio_map = dict(zip(audio["audio_key"], audio["path"]))
    ref_col = schema.get("audio_col") if schema.get("audio_col") in train.columns else id_col
    train["id"] = train[id_col].astype(str)
    train["audio_key"] = train[ref_col].astype(str).map(lambda x: Path(x).stem)
    train["path"] = train["audio_key"].map(audio_map)
    if train["path"].isna().any():
        raise FileNotFoundError(f"train 오디오 매칭 실패 {int(train['path'].isna().sum())}개")

    if len(target_cols) == 1:
        values = train[target_cols[0]].astype(int)
        if not set(values.unique()).issubset({0, 1}):
            raise ValueError("단일 target은 0/1이어야 합니다.")
        train["label"] = 1 - values if schema["single_target_positive"] == "spoof" else values
        label_cols: str | list[str] = "label"  # 내부 규약: 1=bonafide
    else:
        label_cols = target_cols

    group_col = schema.get("group_col")
    indices = np.arange(len(train))
    if group_col and group_col in train.columns:
        splitter = GroupShuffleSplit(n_splits=1, test_size=dev_size, random_state=CFG.seed)
        train_idx, dev_idx = next(splitter.split(indices, groups=train[group_col]))
    else:
        stratify = train[target_cols[0]] if len(target_cols) == 1 else None
        train_idx, dev_idx = train_test_split(
            indices, test_size=dev_size, random_state=CFG.seed, stratify=stratify,
        )
    train_part = train.iloc[train_idx].reset_index(drop=True)
    dev_part = train.iloc[dev_idx].reset_index(drop=True)
    assert set(train_part["id"]).isdisjoint(set(dev_part["id"]))
    return train_part, dev_part, target_cols, label_cols


## 5. EDA — 길이·샘플레이트·채널·RMS·peak·ZCR

전체 파일을 매번 읽지 않도록 먼저 최대 2,000개를 표본 조사합니다. 모델 선택 전에 sample rate 혼합, 무음/클리핑, 길이 꼬리, 클래스/track/attack 분포를 확인합니다.


In [ ]:
def audio_metadata(path: str) -> dict:
    info = sf.info(path)
    audio, sr = sf.read(path, dtype="float32", always_2d=True)
    mono = audio.mean(axis=1)
    rms = float(np.sqrt(np.mean(np.square(mono)) + 1e-12))
    peak = float(np.max(np.abs(mono))) if len(mono) else 0.0
    zcr = float(np.mean(mono[:-1] * mono[1:] < 0)) if len(mono) > 1 else 0.0
    return {
        "duration": info.frames / info.samplerate,
        "sample_rate": info.samplerate,
        "channels": info.channels,
        "rms": rms,
        "peak": peak,
        "zcr": zcr,
    }


def run_eda(frame: pd.DataFrame, max_files: int = 2000) -> pd.DataFrame:
    if frame.empty:
        print("EDA 대상 데이터가 없습니다.")
        return pd.DataFrame()
    sample = frame.sample(min(max_files, len(frame)), random_state=CFG.seed).copy()
    metadata = []
    errors = []
    for row in tqdm(sample.itertuples(index=False), total=len(sample), desc="EDA metadata"):
        try:
            metadata.append({"id": row.id, **audio_metadata(row.path)})
        except Exception as exc:
            errors.append((row.path, repr(exc)))
    meta = pd.DataFrame(metadata)
    result = sample.merge(meta, on="id", how="left")
    if errors:
        print("read errors:", errors[:5])
    display(result.describe(include="all").T)
    return result


eda_df = run_eda(asv_df[asv_df["split"] == "train"] if len(asv_df) else pd.DataFrame())


In [ ]:
if len(eda_df):
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    sns.histplot(data=eda_df, x="duration", hue="track", bins=50, ax=axes[0, 0])
    sns.countplot(data=eda_df, x="sample_rate", ax=axes[0, 1])
    sns.countplot(data=eda_df, x="channels", ax=axes[0, 2])
    sns.histplot(data=eda_df, x="rms", hue="key", bins=50, ax=axes[1, 0])
    sns.histplot(data=eda_df, x="peak", hue="key", bins=50, ax=axes[1, 1])
    sns.histplot(data=eda_df, x="zcr", hue="key", bins=50, ax=axes[1, 2])
    plt.tight_layout()
    plt.show()


def show_examples(frame: pd.DataFrame, n: int = 4) -> None:
    if frame.empty:
        return
    picks = frame.groupby(["track", "key"], group_keys=False).head(1).head(n)
    fig, axes = plt.subplots(len(picks), 2, figsize=(14, 3 * len(picks)), squeeze=False)
    for row_idx, row in enumerate(picks.itertuples(index=False)):
        wav, sr = torchaudio.load(row.path)
        wav = wav.mean(0)
        if sr != CFG.sample_rate:
            wav = torchaudio.functional.resample(wav, sr, CFG.sample_rate)
        axes[row_idx, 0].plot(wav[: CFG.sample_rate * 6].numpy())
        axes[row_idx, 0].set_title(f"{row.track}/{row.key}/{row.id}")
        spec = torch.stft(wav[: CFG.sample_rate * 6], n_fft=512, hop_length=160, return_complex=True).abs()
        axes[row_idx, 1].imshow(torch.log1p(spec).numpy(), origin="lower", aspect="auto")
    plt.tight_layout()
    plt.show()


show_examples(asv_df[asv_df["split"] == "train"] if len(asv_df) else pd.DataFrame())


## 6. RawBoost·통신환경 augmentation

`OfficialRawBoost`는 공식 `RawBoost.py`의 함수를 직접 사용합니다. 학습 데이터에만 확률적으로 적용됩니다. 8 kHz 왕복 resampling·gain·clipping은 별도 통신환경 보강이며, validation/TEST에는 절대 랜덤 augmentation을 적용하지 않습니다.


In [ ]:
def import_rawboost_module():
    path = REPO_ROOT / "rawboost" / "RawBoost.py"
    if not path.exists():
        raise FileNotFoundError("RawBoost repo가 없습니다. 2절 clone 셀을 실행하세요.")
    spec = importlib.util.spec_from_file_location("official_rawboost", path)
    module = importlib.util.module_from_spec(spec)
    assert spec.loader is not None
    spec.loader.exec_module(module)
    return module


class OfficialRawBoost:
    """공식 RawBoost 함수 기반. algo 5=(LnL + ISD), algo 4=(LnL + ISD + SSI)."""

    def __init__(self, probability: float = 0.5, algo: int = 5, sample_rate: int = 16000):
        self.p = probability
        self.algo = algo
        self.sr = sample_rate
        self.rb = import_rawboost_module()

    def _lnl(self, x: np.ndarray) -> np.ndarray:
        return self.rb.LnL_convolutive_noise(
            x, N_f=5, nBands=5, minF=20, maxF=min(8000, self.sr // 2 - 1),
            minBW=100, maxBW=1000, minCoeff=10, maxCoeff=100,
            minG=0, maxG=0, minBiasLinNonLin=5, maxBiasLinNonLin=20, fs=self.sr,
        )

    def _isd(self, x: np.ndarray) -> np.ndarray:
        return self.rb.ISD_additive_noise(x, P=10, g_sd=2)

    def _ssi(self, x: np.ndarray) -> np.ndarray:
        return self.rb.SSI_additive_noise(
            x, SNRmin=10, SNRmax=40, nBands=5, minF=20,
            maxF=min(8000, self.sr // 2 - 1), minBW=100, maxBW=1000,
            minCoeff=10, maxCoeff=100, minG=0, maxG=0, fs=self.sr,
        )

    def __call__(self, waveform: torch.Tensor) -> torch.Tensor:
        if random.random() >= self.p:
            return waveform
        x = waveform.detach().cpu().numpy().astype(np.float64)
        if self.algo in (1, 4, 5, 6):
            x = self._lnl(x)
        if self.algo in (2, 4, 5, 7):
            x = self._isd(x)
        if self.algo in (3, 4, 6, 7):
            x = self._ssi(x)
        if self.algo == 8:
            x = self._lnl(x) + self._isd(x)
            x = self.rb.normWav(x, 0)
        return torch.from_numpy(np.asarray(x, dtype=np.float32))


class CommunicationAugment:
    def __init__(self, probability: float = 0.35, sample_rate: int = 16000):
        self.p = probability
        self.sr = sample_rate

    def __call__(self, x: torch.Tensor) -> torch.Tensor:
        if random.random() >= self.p:
            return x
        choice = random.choice(("telephone", "gain", "clip", "noise"))
        if choice == "telephone":
            x = torchaudio.functional.resample(x, self.sr, 8000)
            x = torchaudio.functional.resample(x, 8000, self.sr)
        elif choice == "gain":
            x = x * (10 ** (random.uniform(-8, 6) / 20))
        elif choice == "clip":
            limit = random.uniform(0.3, 0.9)
            x = torch.clamp(x, -limit, limit) / limit
        else:
            signal_power = x.square().mean().clamp_min(1e-8)
            snr_db = random.uniform(15, 35)
            noise_power = signal_power / (10 ** (snr_db / 10))
            x = x + torch.randn_like(x) * noise_power.sqrt()
        return x.clamp(-1, 1)


## 7. 공통 Dataset / DataLoader

짧은 파일은 AASIST 공식 방식처럼 반복 padding하고, 긴 파일은 train에서 random crop, dev/test에서 deterministic center crop을 씁니다. 제출 추론은 뒤에서 한 파일의 여러 segment를 평균냅니다.


In [ ]:
def load_mono(path: str, target_sr: int = 16000) -> torch.Tensor:
    wav, sr = torchaudio.load(path)
    wav = wav.float().mean(0)
    if wav.numel() == 0:
        raise ValueError(f"빈 오디오: {path}")
    if sr != target_sr:
        wav = torchaudio.functional.resample(wav, sr, target_sr)
    return wav


def fixed_length(wav: torch.Tensor, length: int, train: bool) -> torch.Tensor:
    if wav.numel() >= length:
        if train:
            start = random.randint(0, wav.numel() - length)
        else:
            start = (wav.numel() - length) // 2
        return wav[start : start + length]
    repeats = math.ceil(length / wav.numel())
    return wav.repeat(repeats)[:length]


class AudioFrameDataset(Dataset):
    def __init__(
        self,
        frame: pd.DataFrame,
        label_cols: str | Sequence[str] = "label",
        train: bool = False,
        clip_samples: int = 64_600,
        rawboost_probability: float = 0.0,
        rawboost_algo: int = 5,
        communication_probability: float = 0.0,
    ):
        self.frame = frame.reset_index(drop=True).copy()
        self.label_cols = [label_cols] if isinstance(label_cols, str) else list(label_cols)
        self.train = train
        self.clip_samples = clip_samples
        self.rawboost = (
            OfficialRawBoost(rawboost_probability, rawboost_algo, CFG.sample_rate)
            if rawboost_probability > 0 else None
        )
        self.communication = CommunicationAugment(communication_probability, CFG.sample_rate)

    def __len__(self) -> int:
        return len(self.frame)

    def __getitem__(self, index: int) -> dict:
        row = self.frame.iloc[index]
        wav = load_mono(row["path"], CFG.sample_rate)
        if self.train and self.rawboost is not None:
            wav = self.rawboost(wav)
        if self.train:
            wav = self.communication(wav)
        wav = fixed_length(wav, self.clip_samples, self.train)
        labels = row[self.label_cols].to_numpy(dtype=np.float32)
        if len(self.label_cols) == 1:
            label = torch.tensor(int(labels[0]), dtype=torch.long)
        else:
            label = torch.tensor(labels, dtype=torch.float32)
        return {"audio": wav, "label": label, "id": str(row["id"])}


def balanced_sampler(frame: pd.DataFrame) -> WeightedRandomSampler:
    group_cols = [c for c in ("track", "label") if c in frame.columns]
    keys = frame[group_cols].astype(str).agg("|".join, axis=1)
    counts = keys.value_counts()
    weights = keys.map(lambda key: 1.0 / counts[key]).to_numpy()
    return WeightedRandomSampler(torch.as_tensor(weights, dtype=torch.double), len(weights), replacement=True)


def make_loaders(train_frame: pd.DataFrame, dev_frame: pd.DataFrame, exp: dict):
    label_cols = exp.get("label_cols", "label")
    train_ds = AudioFrameDataset(
        train_frame,
        label_cols=label_cols,
        train=True,
        clip_samples=exp["clip_samples"],
        rawboost_probability=exp.get("rawboost_probability", 0.0),
        rawboost_algo=exp.get("rawboost_algo", 5),
        communication_probability=exp.get("communication_probability", 0.0),
    )
    dev_ds = AudioFrameDataset(
        dev_frame, label_cols=label_cols, train=False, clip_samples=exp["clip_samples"]
    )
    sampler = balanced_sampler(train_frame) if exp.get("balanced", True) else None
    train_loader = DataLoader(
        train_ds, batch_size=exp["batch_size"], sampler=sampler,
        shuffle=sampler is None, num_workers=CFG.num_workers, pin_memory=True,
        drop_last=True,
    )
    dev_loader = DataLoader(
        dev_ds, batch_size=exp.get("eval_batch_size", exp["batch_size"]),
        shuffle=False, num_workers=CFG.num_workers, pin_memory=True,
    )
    return train_loader, dev_loader


## 8. 공통 지표

ASVspoof binary validation의 **1차 선택 지표는 threshold 0.5의 accuracy**입니다. balanced accuracy, F1, EER, AUC는 보조 진단 지표로 기록합니다. Accuracy는 다수 클래스를 전부 맞히는 모델에도 높게 나올 수 있으므로 class-balanced sampler와 domain별 confusion 성능을 반드시 함께 확인합니다. DACON 공식 산식이 공개되면 최종 제출 선택 기준은 공식 지표를 우선합니다.


In [ ]:
def compute_eer(y_true: np.ndarray, score_bonafide: np.ndarray) -> tuple[float, float]:
    fpr, tpr, thresholds = roc_curve(y_true, score_bonafide, pos_label=1)
    fnr = 1 - tpr
    idx = int(np.nanargmin(np.abs(fnr - fpr)))
    return float((fnr[idx] + fpr[idx]) / 2), float(thresholds[idx])


def binary_metrics(y_true: np.ndarray, score_bonafide: np.ndarray) -> dict:
    eer, eer_threshold = compute_eer(y_true, score_bonafide)
    pred = (score_bonafide >= 0.5).astype(int)
    return {
        "eer": eer,
        "eer_threshold": eer_threshold,
        "auc": roc_auc_score(y_true, score_bonafide),
        "accuracy": accuracy_score(y_true, pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, pred),
        "f1": f1_score(y_true, pred, zero_division=0),
    }


def multilabel_metrics(y_true: np.ndarray, probabilities: np.ndarray) -> dict:
    pred = (probabilities >= 0.5).astype(int)
    aucs = []
    for idx in range(y_true.shape[1]):
        if np.unique(y_true[:, idx]).size == 2:
            aucs.append(roc_auc_score(y_true[:, idx], probabilities[:, idx]))
    return {
        "macro_auc": float(np.mean(aucs)) if aucs else float("nan"),
        "label_accuracy": float(np.mean(y_true == pred)),
        "subset_accuracy": accuracy_score(y_true, pred),
        "macro_f1": f1_score(y_true, pred, average="macro", zero_division=0),
    }


## 9. EXP01 — Log-Mel CNN sanity baseline

빠른 파이프라인 검증용입니다. 이 모델이 학습되지 않으면 AASIST로 넘어가지 않습니다.


In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_ch: int, out_ch: int):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.SiLU(),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.SiLU(),
            nn.MaxPool2d(2),
        )

    def forward(self, x):
        return self.block(x)


class LogMelCNN(nn.Module):
    def __init__(self, n_mels: int = 96, dropout: float = 0.25, num_outputs: int = 2):
        super().__init__()
        self.mel = torchaudio.transforms.MelSpectrogram(
            sample_rate=CFG.sample_rate, n_fft=1024, win_length=400,
            hop_length=160, n_mels=n_mels, f_min=20, f_max=7600, power=2,
        )
        self.encoder = nn.Sequential(
            ConvBlock(1, 32), ConvBlock(32, 64), ConvBlock(64, 128),
            nn.AdaptiveAvgPool2d((1, 1)), nn.Flatten(),
        )
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(128, num_outputs))

    def features(self, audio: torch.Tensor) -> torch.Tensor:
        x = torch.log(self.mel(audio).clamp_min(1e-6))
        x = (x - x.mean(dim=(-2, -1), keepdim=True)) / (x.std(dim=(-2, -1), keepdim=True) + 1e-5)
        return self.encoder(x.unsqueeze(1))

    def forward(self, audio: torch.Tensor) -> torch.Tensor:
        return self.head(self.features(audio))


## 10. EXP02–03 — 공식 AASIST / AASIST + RawBoost

AASIST 구조는 공식 저장소 코드를 그대로 import합니다. augmentation만 Dataset에서 교체합니다. `freq_aug`는 RawBoost와 별개의 모델 내부 옵션이며 한 번에 너무 많은 augmentation을 켜지 않도록 기본값은 `False`입니다.


In [ ]:
def load_official_aasist(freq_aug: bool = False) -> nn.Module:
    repo = REPO_ROOT / "aasist"
    config_path = repo / "config" / "AASIST.conf"
    if not config_path.exists():
        raise FileNotFoundError("AASIST repo/config가 없습니다. 2절 clone 셀을 실행하세요.")
    with config_path.open(encoding="utf-8") as file:
        model_config = json.load(file)["model_config"]
    if str(repo) not in sys.path:
        sys.path.insert(0, str(repo))
    from models.AASIST import Model as AASISTModel

    class Wrapper(nn.Module):
        def __init__(self):
            super().__init__()
            self.net = AASISTModel(model_config)
            self.freq_aug = freq_aug

        def forward(self, audio: torch.Tensor) -> torch.Tensor:
            _, logits = self.net(audio, Freq_aug=self.freq_aug and self.training)
            return logits

    return Wrapper()


## 11. EXP04–06 — XLS-R pooling / XLS-R + AASIST-inspired dual graph

`XLSRPoolClassifier`는 SSL 표현의 attentive pooling baseline입니다. `XLSRDualGraphClassifier`는 XLS-R frame 특징에 시간/특징 두 view의 graph-attention을 적용한 **AASIST-inspired** 구현이며, 공식 raw-waveform AASIST와 동일한 모델이라고 부르지 않습니다. 논문 재현이 필요한 경우 뒤의 공식 SSL_Anti-spoofing 별도 런타임 절차를 사용하세요.


In [ ]:
from transformers import AutoModel


class SSLBackbone(nn.Module):
    def __init__(self, model_name: str, freeze: bool = True):
        super().__init__()
        self.ssl = AutoModel.from_pretrained(model_name)
        self.hidden_size = self.ssl.config.hidden_size
        self.freeze_all() if freeze else None

    def freeze_all(self):
        for parameter in self.ssl.parameters():
            parameter.requires_grad = False

    def unfreeze_last_n(self, n: int = 4):
        self.freeze_all()
        encoder = getattr(self.ssl, "encoder", None)
        layers = getattr(encoder, "layers", None)
        if layers is None:
            raise AttributeError("이 backbone에서 encoder.layers를 찾지 못했습니다.")
        for layer in layers[-n:]:
            for parameter in layer.parameters():
                parameter.requires_grad = True

    def forward_features(self, audio: torch.Tensor) -> torch.Tensor:
        if not any(p.requires_grad for p in self.ssl.parameters()):
            with torch.no_grad():
                return self.ssl(audio).last_hidden_state
        return self.ssl(audio).last_hidden_state


class AttentivePool(nn.Module):
    def __init__(self, dim: int):
        super().__init__()
        self.score = nn.Sequential(nn.Linear(dim, dim // 2), nn.Tanh(), nn.Linear(dim // 2, 1))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        weights = torch.softmax(self.score(x), dim=1)
        mean = (x * weights).sum(dim=1)
        variance = (weights * (x - mean[:, None]).square()).sum(dim=1).clamp_min(1e-6)
        return torch.cat([mean, variance.sqrt()], dim=-1)


class XLSRPoolClassifier(SSLBackbone):
    def __init__(self, model_name: str, freeze: bool = True, dropout: float = 0.2, num_outputs: int = 2):
        super().__init__(model_name, freeze)
        self.pool = AttentivePool(self.hidden_size)
        self.head = nn.Sequential(
            nn.LayerNorm(self.hidden_size * 2), nn.Dropout(dropout),
            nn.Linear(self.hidden_size * 2, 256), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(256, num_outputs),
        )

    def forward(self, audio: torch.Tensor) -> torch.Tensor:
        return self.head(self.pool(self.forward_features(audio)))


class GraphBlock(nn.Module):
    def __init__(self, dim: int, heads: int = 4, dropout: float = 0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, heads, dropout=dropout, batch_first=True)
        self.norm2 = nn.LayerNorm(dim)
        self.ff = nn.Sequential(nn.Linear(dim, dim * 4), nn.GELU(), nn.Dropout(dropout), nn.Linear(dim * 4, dim))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        z = self.norm1(x)
        x = x + self.attn(z, z, z, need_weights=False)[0]
        return x + self.ff(self.norm2(x))


class XLSRDualGraphClassifier(SSLBackbone):
    def __init__(self, model_name: str, freeze: bool = True, dim: int = 128, dropout: float = 0.2, num_outputs: int = 2):
        super().__init__(model_name, freeze)
        self.proj = nn.Linear(self.hidden_size, dim)
        self.feature_node_proj = nn.Linear(8, dim)
        self.time_graph = nn.Sequential(GraphBlock(dim), GraphBlock(dim))
        self.feature_graph = nn.Sequential(GraphBlock(dim), GraphBlock(dim))
        self.head = nn.Sequential(
            nn.LayerNorm(dim * 4), nn.Dropout(dropout),
            nn.Linear(dim * 4, 256), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(256, num_outputs),
        )

    def forward(self, audio: torch.Tensor) -> torch.Tensor:
        h = self.proj(self.forward_features(audio))            # [B,T,D]
        time_nodes = F.adaptive_avg_pool1d(h.transpose(1, 2), 64).transpose(1, 2)
        feature_nodes = F.adaptive_avg_pool1d(h.transpose(1, 2), 8)
        feature_nodes = self.feature_node_proj(feature_nodes)  # [B,D,D]
        t = self.time_graph(time_nodes)
        f = self.feature_graph(feature_nodes)
        pooled = torch.cat([t.mean(1), t.amax(1), f.mean(1), f.amax(1)], dim=-1)
        return self.head(pooled)


## 12. EXP07–08 — Bi-Mamba와 음성/음악 멀티브랜치

Mamba는 Colab CUDA/PyTorch 조합에 맞는 wheel 또는 빌드가 필요하므로 선택 설치입니다. 대회 label이 공개되기 전에는 멀티브랜치 출력 수와 의미를 확정할 수 없습니다. `num_outputs`를 `sample_submission`의 target 열 수와 맞춘 뒤 공개 train label로 학습합니다.


In [ ]:
# Mamba 실험을 실행할 런타임에서만 설치:
# %pip install -q "mamba-ssm>=2.2" causal-conv1d


class XLSRBiMambaClassifier(SSLBackbone):
    def __init__(self, model_name: str, freeze: bool = True, dim: int = 256, dropout: float = 0.2, num_outputs: int = 2):
        super().__init__(model_name, freeze)
        try:
            from mamba_ssm import Mamba
        except ImportError as exc:
            raise ImportError("위 선택 설치 셀로 mamba-ssm을 설치한 뒤 런타임을 재시작하세요.") from exc
        self.proj = nn.Linear(self.hidden_size, dim)
        self.forward_mamba = Mamba(d_model=dim, d_state=16, d_conv=4, expand=2)
        self.backward_mamba = Mamba(d_model=dim, d_state=16, d_conv=4, expand=2)
        self.pool = AttentivePool(dim * 2)
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(dim * 4, num_outputs))

    def forward(self, audio: torch.Tensor) -> torch.Tensor:
        h = self.proj(self.forward_features(audio))
        forward = self.forward_mamba(h)
        backward = torch.flip(self.backward_mamba(torch.flip(h, dims=(1,))), dims=(1,))
        return self.head(self.pool(torch.cat([forward, backward], dim=-1)))


class SpeechMusicMultiBranch(SSLBackbone):
    def __init__(self, model_name: str, num_outputs: int, freeze: bool = True, dropout: float = 0.25):
        super().__init__(model_name, freeze)
        self.speech_pool = AttentivePool(self.hidden_size)
        self.mel = torchaudio.transforms.MelSpectrogram(
            sample_rate=CFG.sample_rate, n_fft=1024, hop_length=160, n_mels=128,
        )
        self.music_encoder = nn.Sequential(
            ConvBlock(1, 32), ConvBlock(32, 64), ConvBlock(64, 128),
            nn.AdaptiveAvgPool2d((1, 1)), nn.Flatten(),
        )
        self.fusion = nn.Sequential(
            nn.Linear(self.hidden_size * 2 + 128, 384), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(384, num_outputs),
        )

    def forward(self, audio: torch.Tensor) -> torch.Tensor:
        speech = self.speech_pool(self.forward_features(audio))
        mel = torch.log(self.mel(audio).clamp_min(1e-6))
        music = self.music_encoder(mel.unsqueeze(1))
        return self.fusion(torch.cat([speech, music], dim=-1))


## 13. 실험 registry — 검증을 위한 권장 시작값

“최적 파라미터”는 데이터와 평가 산식 없이 미리 존재하지 않습니다. 아래 값은 Colab T4에서 시작하기 좋은 안전한 값이고, 진짜 선택은 dev/OOF 점수와 Optuna 탐색으로 합니다. LA/PA는 별도 AASIST와 혼합 강건 모델을 모두 남기는 구성이 좋습니다.


In [ ]:
XLSR_MODEL = "facebook/wav2vec2-xls-r-300m"

EXPERIMENTS = {
    "exp001_logmel_lapa": dict(
        model="logmel", tracks=("LA", "PA"), batch_size=32, eval_batch_size=64,
        grad_accum=1, epochs=20, lr=3e-4, backbone_lr=3e-4, weight_decay=1e-4,
        clip_samples=64_600, dropout=0.25, balanced=True, patience=5,
    ),
    "exp002_aasist_la": dict(
        model="aasist", tracks=("LA",), batch_size=8, eval_batch_size=16,
        grad_accum=3, epochs=20, lr=1e-4, backbone_lr=1e-4, weight_decay=1e-4,
        clip_samples=64_600, freq_aug=False, balanced=True, patience=6,
    ),
    "exp002p_aasist_pa": dict(
        model="aasist", tracks=("PA",), batch_size=8, eval_batch_size=16,
        grad_accum=3, epochs=20, lr=1e-4, backbone_lr=1e-4, weight_decay=1e-4,
        clip_samples=64_600, freq_aug=False, balanced=True, patience=6,
    ),
    "exp003_aasist_rawboost_lapa": dict(
        model="aasist", tracks=("LA", "PA"), batch_size=8, eval_batch_size=16,
        grad_accum=3, epochs=24, lr=8e-5, backbone_lr=8e-5, weight_decay=1e-4,
        clip_samples=64_600, rawboost_probability=0.5, rawboost_algo=5,
        communication_probability=0.25, freq_aug=False, balanced=True, patience=7,
    ),
    "exp004_xlsr_pool_lapa": dict(
        model="xlsr_pool", tracks=("LA", "PA"), batch_size=2, eval_batch_size=4,
        grad_accum=8, epochs=10, lr=2e-4, backbone_lr=1e-6, weight_decay=1e-4,
        clip_samples=96_000, freeze=True, freeze_epochs=2, unfreeze_last_n=4,
        dropout=0.20, balanced=True, patience=4,
    ),
    "exp005_xlsr_dualgraph_lapa": dict(
        model="xlsr_dualgraph", tracks=("LA", "PA"), batch_size=2, eval_batch_size=4,
        grad_accum=8, epochs=12, lr=1e-4, backbone_lr=8e-7, weight_decay=1e-4,
        clip_samples=96_000, freeze=True, freeze_epochs=2, unfreeze_last_n=4,
        dropout=0.20, balanced=True, patience=5,
    ),
    "exp006_xlsr_dualgraph_rawboost": dict(
        model="xlsr_dualgraph", tracks=("LA", "PA"), batch_size=2, eval_batch_size=4,
        grad_accum=8, epochs=14, lr=8e-5, backbone_lr=5e-7, weight_decay=1e-4,
        clip_samples=96_000, freeze=True, freeze_epochs=3, unfreeze_last_n=4,
        rawboost_probability=0.35, rawboost_algo=5, communication_probability=0.25,
        dropout=0.25, balanced=True, patience=5,
    ),
    "exp007_xlsr_bimamba": dict(
        model="xlsr_mamba", tracks=("LA", "PA"), batch_size=1, eval_batch_size=2,
        grad_accum=16, epochs=12, lr=8e-5, backbone_lr=5e-7, weight_decay=1e-4,
        clip_samples=96_000, freeze=True, freeze_epochs=3, unfreeze_last_n=4,
        dropout=0.20, balanced=True, patience=5,
    ),
    # 공개 train의 multi-label 전용. target_cols/num_outputs 확정 후 사용.
    "exp008_speech_music_multibranch": dict(
        model="multibranch", tracks=(), batch_size=1, eval_batch_size=2,
        grad_accum=16, epochs=12, lr=1e-4, backbone_lr=5e-7, weight_decay=1e-4,
        clip_samples=96_000, freeze=True, freeze_epochs=3, unfreeze_last_n=4,
        dropout=0.25, balanced=False, patience=5, num_outputs=None,
    ),
}


def build_model(exp: dict) -> nn.Module:
    kind = exp["model"]
    if kind == "logmel":
        return LogMelCNN(dropout=exp.get("dropout", 0.25))
    if kind == "aasist":
        return load_official_aasist(freq_aug=exp.get("freq_aug", False))
    if kind == "xlsr_pool":
        return XLSRPoolClassifier(XLSR_MODEL, freeze=exp.get("freeze", True), dropout=exp.get("dropout", 0.2))
    if kind == "xlsr_dualgraph":
        return XLSRDualGraphClassifier(XLSR_MODEL, freeze=exp.get("freeze", True), dropout=exp.get("dropout", 0.2))
    if kind == "xlsr_mamba":
        return XLSRBiMambaClassifier(XLSR_MODEL, freeze=exp.get("freeze", True), dropout=exp.get("dropout", 0.2))
    if kind == "multibranch":
        if exp.get("num_outputs") is None:
            raise ValueError("대회 target_cols 공개 후 num_outputs를 지정하세요.")
        return SpeechMusicMultiBranch(XLSR_MODEL, exp["num_outputs"], freeze=exp.get("freeze", True), dropout=exp.get("dropout", 0.25))
    raise KeyError(kind)


def load_best_model(experiment_name: str, device: torch.device = DEVICE):
    checkpoint_path = RUN_ROOT / experiment_name / "best.pt"
    checkpoint = torch.load(checkpoint_path, map_location="cpu")
    saved_exp = dict(checkpoint["config"])
    loaded_model = build_model(saved_exp)
    loaded_model.load_state_dict(checkpoint["model_state"], strict=True)
    loaded_model.to(device).eval()
    print("loaded:", checkpoint_path, checkpoint.get("dev_metrics"))
    return loaded_model, saved_exp, checkpoint


## 14. 공통 Trainer — AMP, accumulation, early stopping, resume, dev prediction

체크포인트 선택은 binary ASV 실험에서 dev EER 최소값을 사용합니다. multi-label 대회 train 실험은 macro F1 최대값을 기본으로 하되, 공식 평가식 공개 후 바꿉니다.


In [ ]:
def class_weights_from_frame(frame: pd.DataFrame) -> torch.Tensor:
    counts = frame["label"].value_counts().reindex([0, 1], fill_value=1).to_numpy(dtype=np.float64)
    weights = np.sqrt(counts.sum() / (2 * counts))
    weights = weights / weights.mean()
    return torch.tensor(weights, dtype=torch.float32, device=DEVICE)


def make_optimizer(model: nn.Module, exp: dict) -> torch.optim.Optimizer:
    backbone, head = [], []
    for name, parameter in model.named_parameters():
        (backbone if (name.startswith("ssl.") or name.startswith("net.ssl.")) else head).append(parameter)
    groups = []
    if backbone:
        groups.append({"params": backbone, "lr": exp.get("backbone_lr", exp["lr"])})
    if head:
        groups.append({"params": head, "lr": exp["lr"]})
    return torch.optim.AdamW(groups, weight_decay=exp["weight_decay"])


def probabilities_from_logits(logits: torch.Tensor, multilabel: bool = False) -> torch.Tensor:
    if logits.ndim != 2:
        raise ValueError(f"logits shape 오류: {tuple(logits.shape)}")
    if multilabel:
        return torch.sigmoid(logits)
    if logits.shape[1] != 2:
        raise ValueError(f"binary 모델은 logits 2개가 필요합니다: {tuple(logits.shape)}")
    return torch.softmax(logits, dim=-1)[:, 1]


def run_epoch(model, loader, criterion, optimizer=None, grad_accum=1, scaler=None):
    training = optimizer is not None
    model.train(training)
    total_loss = 0.0
    all_labels, all_probs, all_ids = [], [], []
    if training:
        optimizer.zero_grad(set_to_none=True)

    for step, batch in enumerate(tqdm(loader, leave=False, desc="train" if training else "valid"), 1):
        audio = batch["audio"].to(DEVICE, non_blocking=True)
        label = batch["label"].to(DEVICE, non_blocking=True)
        amp_enabled = DEVICE.type == "cuda"
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=amp_enabled):
            logits = model(audio)
            loss = criterion(logits, label) / (grad_accum if training else 1)

        if training:
            scaler.scale(loss).backward()
            if step % grad_accum == 0 or step == len(loader):
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), 5.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)

        total_loss += float(loss.detach().cpu()) * (grad_accum if training else 1)
        probabilities = probabilities_from_logits(logits.detach(), multilabel=label.ndim > 1).cpu().numpy()
        all_probs.append(probabilities)
        all_labels.append(label.detach().cpu().numpy())
        all_ids.extend(batch["id"])

    y_true = np.concatenate(all_labels)
    probabilities = np.concatenate(all_probs)
    metrics = binary_metrics(y_true, probabilities) if y_true.ndim == 1 else multilabel_metrics(y_true, probabilities)
    metrics["loss"] = total_loss / max(1, len(loader))
    return metrics, y_true, probabilities, all_ids


def fit_model(model, train_loader, dev_loader, train_frame, exp_name: str, exp: dict):
    run_dir = RUN_ROOT / exp_name
    run_dir.mkdir(parents=True, exist_ok=True)
    model = model.to(DEVICE)
    label_cols = exp.get("label_cols", "label")
    multilabel = not isinstance(label_cols, str) and len(label_cols) > 1
    if multilabel:
        positives = train_frame[list(label_cols)].sum(axis=0).to_numpy(dtype=np.float64)
        negatives = len(train_frame) - positives
        pos_weight = torch.tensor(negatives / np.clip(positives, 1, None), dtype=torch.float32, device=DEVICE)
        criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    else:
        weights = None if exp.get("balanced", True) else class_weights_from_frame(train_frame)
        criterion = nn.CrossEntropyLoss(weight=weights)
    optimizer = make_optimizer(model, exp)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=exp["epochs"], eta_min=1e-7)
    scaler = torch.cuda.amp.GradScaler(enabled=DEVICE.type == "cuda")
    history, best, stale = [], -float("inf"), 0

    for epoch in range(1, exp["epochs"] + 1):
        if epoch == exp.get("freeze_epochs", -1) + 1 and hasattr(model, "unfreeze_last_n"):
            model.unfreeze_last_n(exp.get("unfreeze_last_n", 4))
            print("unfroze last SSL layers")
        train_metrics, *_ = run_epoch(
            model, train_loader, criterion, optimizer,
            grad_accum=exp.get("grad_accum", 1), scaler=scaler,
        )
        with torch.no_grad():
            dev_metrics, y_dev, p_dev, ids_dev = run_epoch(model, dev_loader, criterion)
        scheduler.step()
        record = {"epoch": epoch, **{f"train_{k}": v for k, v in train_metrics.items()}, **{f"dev_{k}": v for k, v in dev_metrics.items()}}
        history.append(record)
        print(json.dumps(record, ensure_ascii=False))
        pd.DataFrame(history).to_csv(run_dir / "history.csv", index=False)

        score = dev_metrics["label_accuracy"] if multilabel else dev_metrics["accuracy"]
        improved = score > best
        if improved:
            best, stale = score, 0
            torch.save({
                "model_state": model.state_dict(), "experiment": exp_name,
                "config": exp, "project_config": asdict(CFG), "repo_commits": REPO_COMMITS,
                "dev_metrics": dev_metrics, "label_convention": {"0": "spoof", "1": "bonafide"},
            }, run_dir / "best.pt")
            if multilabel:
                pred_frame = pd.DataFrame({"id": ids_dev})
                for idx, column in enumerate(label_cols):
                    pred_frame[f"label_{column}"] = y_dev[:, idx]
                    pred_frame[f"score_{column}"] = p_dev[:, idx]
            else:
                pred_frame = pd.DataFrame({"id": ids_dev, "label": y_dev, "score_bonafide": p_dev})
            pred_frame.to_csv(run_dir / "dev_predictions.csv", index=False)
        else:
            stale += 1
        if stale >= exp["patience"]:
            print("early stopping")
            break
    return pd.DataFrame(history), run_dir / "best.pt"


## 15. 실험 선택 및 학습

처음에는 `exp001_logmel_lapa`를 1 epoch로 바꿔 end-to-end smoke test한 뒤 정상 종료되면 원래 epoch로 돌립니다. 모델별로 런타임을 새로 시작하면 GPU 메모리 파편화를 줄일 수 있습니다.


In [ ]:
SELECTED_EXPERIMENT = "exp001_logmel_lapa"
RUN_TRAINING = False  # 데이터 경로와 shape 확인 후 True

exp = dict(EXPERIMENTS[SELECTED_EXPERIMENT])
print(json.dumps(exp, indent=2, ensure_ascii=False))

if RUN_TRAINING:
    if exp["model"] == "multibranch":
        schema = infer_schema(dacon_contract, DACON_SCHEMA)
        train_frame, dev_frame, target_cols, label_cols = prepare_dacon_supervised_frames(dacon_contract, schema)
        exp["label_cols"] = label_cols
        exp["num_outputs"] = 2 if len(target_cols) == 1 else len(target_cols)
    else:
        if asv_df.empty:
            raise RuntimeError("ASVspoof index가 비었습니다. ZIP 경로/압축 구조를 확인하세요.")
        selected = asv_df[asv_df["track"].isin(exp["tracks"])].copy()
        train_frame = selected[selected["split"] == "train"].reset_index(drop=True)
        dev_frame = selected[selected["split"] == "dev"].reset_index(drop=True)
        assert len(train_frame) and len(dev_frame)
        assert set(train_frame["id"]).isdisjoint(set(dev_frame["id"])), "train/dev leakage"

    train_loader, dev_loader = make_loaders(train_frame, dev_frame, exp)
    sanity = next(iter(train_loader))
    print("audio:", sanity["audio"].shape, "label:", sanity["label"].shape)
    model = build_model(exp)
    with torch.no_grad():
        test_logits = model(sanity["audio"][:2].to(DEVICE))
    print("logits:", test_logits.shape)
    expected_outputs = exp.get("num_outputs") or 2
    assert test_logits.shape == (min(2, len(sanity["audio"])), expected_outputs)

    history, best_path = fit_model(model, train_loader, dev_loader, train_frame, SELECTED_EXPERIMENT, exp)
    print("best checkpoint:", best_path)


## 16. ASVspoof 2019 공식 eval 평가

하이퍼파라미터와 ensemble 가중치는 dev/OOF accuracy로만 결정합니다. 설정을 고정한 뒤 공식 eval split은 최종 비교용으로 한 번 평가합니다. LA/PA별 accuracy와 balanced accuracy를 따로 기록해 혼합 평균이 한 domain의 실패를 가리지 않게 하며, EER도 보조 지표로 남깁니다.


In [ ]:
@torch.no_grad()
def evaluate_asvspoof_eval(model: nn.Module, frame: pd.DataFrame, exp: dict, experiment_name: str):
    output_rows = []
    metrics_by_track = {}
    criterion = nn.CrossEntropyLoss()
    for track in exp["tracks"]:
        eval_frame = frame[(frame["split"] == "eval") & (frame["track"] == track)].reset_index(drop=True)
        if eval_frame.empty:
            print(f"[skip] {track} eval 없음")
            continue
        dataset = AudioFrameDataset(eval_frame, train=False, clip_samples=exp["clip_samples"])
        loader = DataLoader(
            dataset, batch_size=exp.get("eval_batch_size", exp["batch_size"]),
            shuffle=False, num_workers=CFG.num_workers, pin_memory=True,
        )
        metrics, labels, probabilities, identifiers = run_epoch(model, loader, criterion)
        metrics_by_track[track] = metrics
        output_rows.append(pd.DataFrame({
            "id": identifiers, "track": track, "label": labels,
            "score_bonafide": probabilities,
        }))
        print(track, metrics)
    if output_rows:
        output = pd.concat(output_rows, ignore_index=True)
        run_dir = RUN_ROOT / experiment_name
        output.to_csv(run_dir / "eval_predictions.csv", index=False)
        (run_dir / "eval_metrics.json").write_text(
            json.dumps(metrics_by_track, indent=2, ensure_ascii=False), encoding="utf-8"
        )
        return metrics_by_track, output
    return {}, pd.DataFrame()


RUN_ASVSPOOF_EVAL = False
if RUN_ASVSPOOF_EVAL:
    model, exp, checkpoint = load_best_model(SELECTED_EXPERIMENT)
    eval_metrics, eval_predictions = evaluate_asvspoof_eval(
        model, asv_df, exp, SELECTED_EXPERIMENT
    )


## 17. 하이퍼파라미터 탐색 원칙

TEST가 아니라 dev/OOF에서만 탐색합니다. 아래 search space를 기준으로 2–3 epoch의 저비용 trial을 거친 뒤, 상위 2–3개 설정만 full epoch로 재학습하세요. 서로 다른 seed까지 비교하지 않으면 우연한 차이를 “최적”으로 오인하기 쉽습니다.

권장 탐색 범위:

| 모델 | lr(head) | lr(backbone) | weight decay | 기타 |
|---|---:|---:|---:|---|
| LogMel | 1e-4–1e-3 | 동일 | 1e-6–1e-3 | n_mels 80/96/128, dropout .1–.4 |
| AASIST | 3e-5–3e-4 | 동일 | 1e-6–5e-4 | RawBoost p 0/.25/.5, algo 5/8 |
| XLS-R | 5e-5–3e-4 | 1e-7–3e-6 | 1e-6–5e-4 | freeze 1–4 epochs, last 2/4/6 layers |
| Mamba | 3e-5–2e-4 | 1e-7–1e-6 | 1e-6–5e-4 | d_model 128/256, d_state 8/16 |


In [ ]:
import optuna

RUN_OPTUNA = False
OPTUNA_BASE_EXPERIMENT = "exp001_logmel_lapa"


def optuna_objective(trial: optuna.Trial) -> float:
    base = dict(EXPERIMENTS[OPTUNA_BASE_EXPERIMENT])
    base.update({
        "lr": trial.suggest_float("lr", 1e-4, 1e-3, log=True),
        "weight_decay": trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True),
        "dropout": trial.suggest_float("dropout", 0.1, 0.4),
        "epochs": 3,
        "patience": 3,
    })
    selected = asv_df[asv_df["track"].isin(base["tracks"])]
    train_frame = selected[selected["split"] == "train"].sample(frac=0.25, random_state=CFG.seed)
    dev_frame = selected[selected["split"] == "dev"].sample(frac=0.5, random_state=CFG.seed)
    train_loader, dev_loader = make_loaders(train_frame, dev_frame, base)
    model = build_model(base)
    history, _ = fit_model(model, train_loader, dev_loader, train_frame, f"optuna_trial_{trial.number:03d}", base)
    value = float(history["dev_accuracy"].max())
    del model
    gc.collect()
    torch.cuda.empty_cache()
    return value


if RUN_OPTUNA:
    study = optuna.create_study(direction="maximize", study_name="deepvoice_dev_accuracy")
    study.optimize(optuna_objective, n_trials=20)
    print(study.best_value, study.best_params)


## 18. 공식 SSL_Anti-spoofing / XLSR-Mamba 논문 재현(별도 Colab 런타임)

두 공식 코드는 pinned legacy fairseq와 오래된 PyTorch 환경을 전제로 합니다. 현재 transformers 파이프라인과 같은 런타임에 강제로 섞으면 의존성 충돌 가능성이 큽니다.

1. `CLONE_LEGACY_ADVANCED_REPOS=True`로 clone하고 commit을 기록합니다.
2. 새 Colab 런타임에서 각 저장소 README의 fairseq commit과 requirements를 설치합니다.
3. ASVspoof 2019 LA train/dev 경로로 먼저 공식 command를 재현합니다.
4. 공식 checkpoint와 dev score 파일을 `RUN_ROOT/exp007_official_*`에 복사합니다.
5. 아래 ensemble 형식(`id,label,score_bonafide`)으로 score를 변환합니다.

공식 SSL 구현의 공개 시작값은 LA에서 `lr=1e-6`, `batch_size=14`, weighted CE입니다. T4 메모리가 부족하면 batch 2와 accumulation 7을 사용합니다. 공식 XLSR-Mamba는 fixed-length input의 `--algo 5`를 제시합니다. 구현·가중치 출처와 license는 반드시 2차 보고서에 기록하세요.


## 19. Dev/OOF 기반 ensemble, calibration, threshold

가중치와 단일 global threshold는 dev/OOF accuracy에서 결정합니다. Domain마다 별도 threshold를 맞추면 실제 일반화 성능을 과대평가하므로 사용하지 않습니다. TEST 예측끼리 통계적으로 보정하지 않으며, 공식 제출이 확률을 요구하면 hard threshold를 적용하지 않고 calibration된 확률을 제출합니다.


In [ ]:
def load_prediction_files(paths: Sequence[str | Path]) -> tuple[pd.DataFrame, np.ndarray]:
    frames = [pd.read_csv(path).rename(columns={"score_bonafide": f"m{i}"}) for i, path in enumerate(paths)]
    merged = frames[0]
    for i, frame in enumerate(frames[1:], 1):
        merged = merged.merge(frame[["id", f"m{i}"]], on="id", validate="one_to_one")
    model_cols = [c for c in merged if c.startswith("m")]
    return merged, merged[model_cols].to_numpy()


def optimize_accuracy_weights(
    y_true: np.ndarray, predictions: np.ndarray,
    threshold: float = 0.5, n_candidates: int = 5000,
) -> np.ndarray:
    """OOF accuracy를 직접 최대화하고 동률이면 log-loss가 낮은 weight를 선택합니다."""
    n_models = predictions.shape[1]
    rng = np.random.default_rng(CFG.seed)
    candidates = [np.full(n_models, 1 / n_models), *np.eye(n_models)]
    candidates.extend(rng.dirichlet(np.ones(n_models), size=n_candidates))
    best_weight, best_key = candidates[0], (-float("inf"), -float("inf"))
    for weights in candidates:
        score = np.clip(predictions @ weights, 1e-6, 1 - 1e-6)
        key = (
            accuracy_score(y_true, score >= threshold),
            -log_loss(y_true, score, labels=[0, 1]),
        )
        if key > best_key:
            best_weight, best_key = np.asarray(weights), key
    return best_weight


def optimize_multilabel_weights(y_true: np.ndarray, predictions: np.ndarray) -> np.ndarray:
    """predictions: [N, M, C]. 각 target C마다 OOF 기반 model weight를 학습."""
    if predictions.ndim != 3 or y_true.shape != (predictions.shape[0], predictions.shape[2]):
        raise ValueError((y_true.shape, predictions.shape))
    return np.stack([
        optimize_accuracy_weights(y_true[:, target], predictions[:, :, target])
        for target in range(y_true.shape[1])
    ])


def apply_multilabel_weights(predictions: np.ndarray, weights: np.ndarray) -> np.ndarray:
    return np.einsum("nmc,cm->nc", predictions, weights)


def fit_platt(y_true: np.ndarray, scores: np.ndarray) -> LogisticRegression:
    logits = np.log(np.clip(scores, 1e-6, 1 - 1e-6) / np.clip(1 - scores, 1e-6, 1))
    return LogisticRegression(C=1.0).fit(logits.reshape(-1, 1), y_true)


def optimize_accuracy_threshold(y_true: np.ndarray, scores: np.ndarray) -> tuple[float, float]:
    """모든 score를 매번 재비교하지 않고 O(N log N)으로 exact accuracy threshold를 찾습니다."""
    y_true = np.asarray(y_true, dtype=np.int64)
    scores = np.asarray(scores, dtype=np.float64)
    if len(y_true) == 0 or len(y_true) != len(scores):
        raise ValueError((y_true.shape, scores.shape))
    order = np.argsort(-scores, kind="mergesort")
    sorted_scores, sorted_labels = scores[order], y_true[order]
    correct = int(np.sum(sorted_labels == 0))  # threshold > max(score): 모두 0 예측
    best_correct = correct
    best_threshold = float(np.nextafter(sorted_scores[0], np.inf))
    index = 0
    while index < len(sorted_scores):
        end = index + 1
        while end < len(sorted_scores) and sorted_scores[end] == sorted_scores[index]:
            end += 1
        group = sorted_labels[index:end]
        correct += int(np.sum(group == 1) - np.sum(group == 0))
        threshold = float(sorted_scores[index])
        if correct > best_correct or (correct == best_correct and abs(threshold - 0.5) < abs(best_threshold - 0.5)):
            best_correct, best_threshold = correct, threshold
        index = end
    return best_threshold, float(best_correct / max(1, len(sorted_labels)))


def optimize_f1_threshold(y_true: np.ndarray, scores: np.ndarray) -> tuple[float, float]:
    candidates = np.linspace(0.02, 0.98, 193)
    values = np.array([f1_score(y_true, scores >= threshold, zero_division=0) for threshold in candidates])
    index = int(values.argmax())
    return float(candidates[index]), float(values[index])


DEV_PREDICTION_FILES = []  # 각 실험의 RUN_ROOT/.../dev_predictions.csv
if DEV_PREDICTION_FILES:
    merged_dev, dev_matrix = load_prediction_files(DEV_PREDICTION_FILES)
    selection_labels = merged_dev["label"].to_numpy()
    decision_threshold = 0.5
    # weight와 global threshold를 번갈아 갱신하는 간단한 coordinate search입니다.
    for _ in range(2):
        ensemble_weights = optimize_accuracy_weights(selection_labels, dev_matrix, threshold=decision_threshold)
        raw_ensemble = dev_matrix @ ensemble_weights
        decision_threshold, _ = optimize_accuracy_threshold(selection_labels, raw_ensemble)
    calibrator = fit_platt(merged_dev["label"].to_numpy(), raw_ensemble)
    raw_logit = np.log(np.clip(raw_ensemble, 1e-6, 1 - 1e-6) / np.clip(1 - raw_ensemble, 1e-6, 1))
    calibrated = calibrator.predict_proba(raw_logit.reshape(-1, 1))[:, 1]
    raw_threshold, raw_accuracy = optimize_accuracy_threshold(merged_dev["label"].to_numpy(), raw_ensemble)
    calibrated_threshold, calibrated_accuracy = optimize_accuracy_threshold(merged_dev["label"].to_numpy(), calibrated)
    print("weights:", ensemble_weights)
    print("raw:", binary_metrics(merged_dev["label"].to_numpy(), raw_ensemble), "threshold/accuracy:", raw_threshold, raw_accuracy)
    print("calibrated:", binary_metrics(merged_dev["label"].to_numpy(), calibrated), "threshold/accuracy:", calibrated_threshold, calibrated_accuracy)


## 20. DACON TEST 독립 추론

각 오디오 안에서 최대 5개 deterministic segment를 평균합니다. 이것은 공식 규칙상 허용된 file-internal segment inference입니다. `sample_submission.csv`의 행 순서를 보존하고, 단일 target일 때 `single_target_positive` 방향을 명시적으로 적용합니다.


In [ ]:
def infer_schema(contract: dict, overrides: dict) -> dict:
    sample = contract.get("sample")
    test = contract.get("test")
    if sample is None:
        raise FileNotFoundError("sample_submission.csv가 없습니다. 8월 26일 공개 데이터를 확인하세요.")
    id_col = overrides.get("id_col") or sample.columns[0]
    target_cols = overrides.get("target_cols") or [c for c in sample.columns if c != id_col]
    audio_col = overrides.get("audio_col")
    if audio_col is None and test is not None:
        candidates = [c for c in test.columns if any(k in c.lower() for k in ("path", "file", "audio", "name"))]
        audio_col = candidates[0] if candidates else None
    return {**overrides, "id_col": id_col, "target_cols": target_cols, "audio_col": audio_col}


def resolve_dacon_test_frame(contract: dict, schema: dict) -> pd.DataFrame:
    sample = contract["sample"].copy()
    test = contract.get("test")
    audio = contract["audio"].copy()
    id_col = schema["id_col"]
    sample["_join_key"] = sample[id_col].astype(str).map(lambda x: Path(x).stem)
    if test is not None and schema["audio_col"] is not None:
        test = test.copy()
        refs = test[schema["audio_col"]].astype(str)
        test["id"] = test[id_col].astype(str) if id_col in test else refs
        test["_join_key"] = test["id"].map(lambda x: Path(x).stem)
        by_stem = dict(zip(audio["id"].astype(str), audio["path"]))
        test["path"] = refs.map(lambda x: str(DACON_ROOT / x) if (DACON_ROOT / x).exists() else by_stem.get(Path(x).stem, ""))
        frame = sample[[id_col, "_join_key"]].merge(test[["id", "_join_key", "path"]], on="_join_key", how="left")
    else:
        audio["_join_key"] = audio["id"].astype(str).map(lambda x: Path(x).stem)
        frame = sample[[id_col, "_join_key"]].merge(audio, on="_join_key", how="left")
    if frame["path"].isna().any() or (frame["path"] == "").any():
        raise FileNotFoundError("sample_submission 행과 TEST 오디오를 모두 매칭하지 못했습니다.")
    return frame


def deterministic_segments(wav: torch.Tensor, length: int, count: int = 5) -> torch.Tensor:
    if wav.numel() <= length:
        return fixed_length(wav, length, train=False).unsqueeze(0)
    starts = np.linspace(0, wav.numel() - length, num=count, dtype=int)
    return torch.stack([wav[start : start + length] for start in starts])


@torch.no_grad()
def predict_files(model: nn.Module, frame: pd.DataFrame, clip_samples: int, segments: int = 5, multilabel: bool = False) -> np.ndarray:
    model.eval().to(DEVICE)
    output = []
    for row in tqdm(frame.itertuples(index=False), total=len(frame), desc="DACON inference"):
        wav = load_mono(row.path, CFG.sample_rate)
        clips = deterministic_segments(wav, clip_samples, segments)
        logits = model(clips.to(DEVICE))
        probs = probabilities_from_logits(logits, multilabel=multilabel).mean(dim=0).cpu().numpy()
        output.append(np.atleast_1d(probs))
    return np.stack(output)


def make_submission(contract: dict, schema: dict, probabilities: np.ndarray, output_path: Path) -> pd.DataFrame:
    sample = contract["sample"].copy()
    targets = schema["target_cols"]
    if probabilities.shape[1] == 1 and len(targets) == 1:
        p_bonafide = probabilities[:, 0]
        values = 1 - p_bonafide if schema["single_target_positive"] == "spoof" else p_bonafide
        sample[targets[0]] = values
    elif probabilities.shape[1] == len(targets):
        sample.loc[:, targets] = probabilities
    else:
        raise ValueError(f"model outputs={probabilities.shape[1]}, submission targets={len(targets)}")
    if sample[targets].isna().any().any():
        raise ValueError("submission에 NaN이 있습니다.")
    output_path.parent.mkdir(parents=True, exist_ok=True)
    sample.to_csv(output_path, index=False, encoding="utf-8")
    return sample


In [ ]:
RUN_DACON_INFERENCE = False

if RUN_DACON_INFERENCE:
    schema = infer_schema(dacon_contract, DACON_SCHEMA)
    print(schema)
    test_frame = resolve_dacon_test_frame(dacon_contract, schema)
    # model은 15절에서 학습했거나 best.pt를 build_model(exp)에 load한 객체여야 합니다.
    test_probabilities = predict_files(
        model, test_frame, exp["clip_samples"], segments=5,
        multilabel=len(schema["target_cols"]) > 1,
    )
    submission = make_submission(
        dacon_contract, schema, test_probabilities,
        EXPORT_ROOT / "local_submission.csv",
    )
    display(submission.head())


## 21. 코드 제출용 `submit.zip` 생성

아래 exporter는 최종 모델을 fixed-length TorchScript로 저장하고, 오프라인 `script.py`가 `/data`(또는 ZIP 옆 `data`)를 읽어 `output/submission.csv`를 생성하게 합니다. 대회 공개 후 반드시 서버 기본 패키지·용량·제한 시간에 맞춰 smoke test하세요. XLS-R 300M ensemble은 ZIP 제한을 초과할 수 있으므로 distillation 또는 모델 수 축소가 필요할 수 있습니다.


In [ ]:
SUBMISSION_SCRIPT = r'''
from pathlib import Path
import json
import math
import numpy as np
import pandas as pd
import soundfile as sf
import torch
from scipy.signal import resample_poly

BASE = Path(__file__).resolve().parent
DATA = BASE / "data" if (BASE / "data").exists() else Path("/data")
OUTPUT = BASE / "output"
OUTPUT.mkdir(parents=True, exist_ok=True)
META = json.loads((BASE / "model" / "metadata.json").read_text(encoding="utf-8"))
MODEL = torch.jit.load(str(BASE / "model" / "model.ts"), map_location="cpu").eval()
SUFFIXES = {".wav", ".flac", ".ogg", ".mp3", ".m4a", ".aac"}

def load_mono(path):
    x, sr = sf.read(path, dtype="float32", always_2d=True)
    x = x.mean(axis=1)
    if sr != META["sample_rate"]:
        divisor = math.gcd(sr, META["sample_rate"])
        x = resample_poly(x, META["sample_rate"] // divisor, sr // divisor).astype("float32")
    if len(x) == 0:
        raise ValueError(f"empty audio: {path}")
    return torch.from_numpy(x)

def segments(x):
    length = META["clip_samples"]
    if len(x) <= length:
        x = x.repeat(math.ceil(length / len(x)))[:length]
        return x.unsqueeze(0)
    starts = np.linspace(0, len(x) - length, META["segments"], dtype=int)
    return torch.stack([x[s:s+length] for s in starts])

sample_paths = sorted(DATA.rglob("*sample*submission*.csv"))
if len(sample_paths) != 1:
    raise FileNotFoundError(f"sample_submission count={len(sample_paths)}")
sample = pd.read_csv(sample_paths[0])
id_col = META["id_col"]
targets = META["target_cols"]
audio = {p.stem: p for p in DATA.rglob("*") if p.is_file() and p.suffix.lower() in SUFFIXES}
predictions = []
with torch.inference_mode():
    for identifier in sample[id_col].astype(str):
        audio_key = Path(identifier).stem
        if audio_key not in audio:
            raise FileNotFoundError(f"audio not found: {identifier}")
        predictions.append(MODEL(segments(load_mono(audio[audio_key]))).mean(0).numpy())
predictions = np.stack(predictions)
if predictions.ndim == 1:
    predictions = predictions[:, None]
if predictions.shape[1] != len(targets):
    raise ValueError((predictions.shape, targets))
sample.loc[:, targets] = predictions
if sample[targets].isna().any().any():
    raise ValueError("NaN in submission")
sample.to_csv(OUTPUT / "submission.csv", index=False, encoding="utf-8")
'''


class SubmissionProbabilityWrapper(nn.Module):
    def __init__(self, base_model: nn.Module, num_targets: int, single_target_positive: str = "spoof"):
        super().__init__()
        self.base_model = base_model
        self.num_targets = num_targets
        self.spoof_positive = single_target_positive == "spoof"

    def forward(self, audio: torch.Tensor) -> torch.Tensor:
        logits = self.base_model(audio)
        if self.num_targets == 1:
            if logits.shape[-1] != 2:
                raise RuntimeError("single target binary 모델은 logits 2개가 필요합니다.")
            p_bonafide = torch.softmax(logits, dim=-1)[:, 1:2]
            return 1 - p_bonafide if self.spoof_positive else p_bonafide
        return torch.sigmoid(logits)


def build_submit_zip(model: nn.Module, schema: dict, exp: dict, destination: Path) -> Path:
    package = WORK_ROOT / "submit_package"
    if package.exists():
        shutil.rmtree(package)
    model_dir = package / "model"
    model_dir.mkdir(parents=True)
    wrapper = SubmissionProbabilityWrapper(
        model.eval().cpu(), len(schema["target_cols"]), schema["single_target_positive"]
    )
    example = torch.zeros(1, exp["clip_samples"])
    traced = torch.jit.trace(wrapper, example, strict=False)
    traced.save(str(model_dir / "model.ts"))
    metadata = {
        "sample_rate": CFG.sample_rate, "clip_samples": exp["clip_samples"],
        "segments": 5, "id_col": schema["id_col"], "target_cols": schema["target_cols"],
        "single_target_positive": schema["single_target_positive"],
        "experiment": SELECTED_EXPERIMENT, "repo_commits": REPO_COMMITS,
    }
    (model_dir / "metadata.json").write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8")
    (package / "script.py").write_text(SUBMISSION_SCRIPT, encoding="utf-8")
    (package / "requirements.txt").write_text("soundfile\nscipy\npandas\n", encoding="utf-8")

    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists():
        destination.unlink()
    shutil.make_archive(str(destination.with_suffix("")), "zip", package)
    with zipfile.ZipFile(destination) as zf:
        names = set(zf.namelist())
        assert "script.py" in names and "requirements.txt" in names
        assert "model/model.ts" in names and "model/metadata.json" in names
    print("created:", destination, "MB=", destination.stat().st_size / 1024**2)
    return destination


BUILD_SUBMIT_ZIP = False
if BUILD_SUBMIT_ZIP:
    schema = infer_schema(dacon_contract, DACON_SCHEMA)
    submit_zip = build_submit_zip(model, schema, exp, EXPORT_ROOT / "submit.zip")


## 22. 제출 전 필수 smoke test

1. 대회 공개 후 평가식·target 의미·sample submission 열을 `DACON_SCHEMA`에 확정한다.
2. TEST는 어떤 학습·튜닝·pseudo-label에도 사용하지 않는다.
3. best checkpoint를 새 런타임에서 load해 dev 점수가 재현되는지 확인한다.
4. `submit.zip`을 임시 폴더에 풀고 인터넷을 사용하지 않은 상태에서 `python script.py`를 실행한다.
5. 결과가 정확히 `output/submission.csv`, UTF-8, sample과 동일한 행 순서/열 순서/행 수인지 확인한다.
6. 각 TEST 파일 prediction이 다른 TEST 파일의 통계나 prediction에 의존하지 않는지 확인한다.
7. ZIP 루트에 추가 상위 폴더가 없는지, 용량·설치·추론 제한을 충족하는지 확인한다.
8. 사용한 ASVspoof 파일, pretrained weight, 저장소 URL·commit·license를 학습데이터 구성 보고서에 기록한다.

### 권장 실제 순서

`EXP01 smoke → EXP02 LA → EXP02P PA → EXP03 mixed RawBoost → EXP04 SSL pooling → EXP05/06 dual graph → EXP07 Mamba → (공개 label 확인 후) EXP08 multi-branch → OOF ensemble/calibration → offline submit.zip smoke test`

단계별 결과는 `MyDrive/deepvoice_dacon/runs/<experiment>/history.csv`, `best.pt`, `dev_predictions.csv`에 남습니다.


# Part B. 교차 데이터셋 강건성 확장

앞부분은 실행 가능한 공통 기반입니다. 이 Part B는 다음 기술을 결합합니다.

- 데이터셋별 표준 manifest와 라이선스 승인 gate
- 단일 dev가 아닌 cross-dev checkpoint 선택
- untouched audit set을 이용한 최종 교차 데이터셋 평가
- bitrate·silence·고주파·phase 통계 EDA
- LFCC + multi-scale Log-Mel ConvNeXt-style branch
- 실제 FFmpeg codec cache, RIR, conservative waveform augmentation
- WavLM/XLS-R + Temporal-Channel Modeling
- 평균 EER뿐 아니라 worst-domain EER와 domain 편차를 반영한 robust score
- 여러 체크포인트를 `model/`에 배치하는 정확한 DACON ensemble `submit.zip`

`audit` 데이터는 checkpoint·threshold·ensemble weight 선택에 사용하지 않습니다. 모든 설정을 확정한 후 마지막 보고서 생성에만 사용합니다.


## B1. 표준 manifest와 domain 역할

모든 외부 데이터셋은 다음 열로 통일합니다.

`id, path, label, dataset, stage, speaker, generator, codec, content_type`

- `stage=train`: gradient update에 사용
- `stage=selection`: epoch/checkpoint/파라미터 선택에 사용
- `stage=audit`: 모든 설정을 고정한 뒤 한 번만 평가

외부 데이터는 `license_approved=True`로 명시한 경우에만 train에 들어갑니다. 출처가 불명확한 데이터는 평가용이라도 사용 조건을 먼저 확인하세요.


In [ ]:
from dataclasses import dataclass
from sklearn.model_selection import StratifiedGroupKFold

STANDARD_MANIFEST_COLUMNS = [
    "id", "path", "label", "dataset", "stage",
    "speaker", "generator", "codec", "content_type",
]


@dataclass
class ExternalDatasetSpec:
    name: str
    manifest_path: str
    stage: str
    license_approved: bool
    source_url: str


# manifest_path를 Drive의 실제 파일로 수정합니다.
EXTERNAL_DATASETS = [
    ExternalDatasetSpec("ASVspoof2021_DF", str(DRIVE_ROOT / "manifests/asv2021_df.csv"), "audit", True, "https://www.asvspoof.org/index2021.html"),
    ExternalDatasetSpec("InTheWild", str(DRIVE_ROOT / "manifests/in_the_wild.csv"), "audit", False, "https://huggingface.co/datasets/mueller91/In-The-Wild"),
    ExternalDatasetSpec("MLAAD", str(DRIVE_ROOT / "manifests/mlaad_selection.csv"), "selection", False, "https://arxiv.org/abs/2401.09512"),
    ExternalDatasetSpec("Codecfake_C7", str(DRIVE_ROOT / "manifests/codecfake_c7.csv"), "audit", False, "https://github.com/xieyuankun/Codecfake"),
    ExternalDatasetSpec("ReplayDF", str(DRIVE_ROOT / "manifests/replaydf.csv"), "audit", False, "https://arxiv.org/abs/2505.14862"),
    ExternalDatasetSpec("EnvSDD", str(DRIVE_ROOT / "manifests/envsdd.csv"), "audit", False, "https://github.com/apple-yinhan/EnvSDD"),
    ExternalDatasetSpec("SONICS", str(DRIVE_ROOT / "manifests/sonics.csv"), "audit", False, "https://arxiv.org/abs/2408.14080"),
]


def normalize_manifest(frame: pd.DataFrame, dataset: str, stage: str) -> pd.DataFrame:
    frame = frame.copy()
    required = {"id", "path", "label"}
    missing = required - set(frame.columns)
    if missing:
        raise KeyError(f"{dataset} manifest 필수 열 누락: {sorted(missing)}")
    frame["dataset"] = dataset
    frame["stage"] = stage
    frame["label"] = frame["label"].astype(int)
    if not set(frame["label"].unique()).issubset({0, 1}):
        raise ValueError(f"{dataset}: label은 내부 규약 0=spoof, 1=bonafide여야 합니다.")
    for column in STANDARD_MANIFEST_COLUMNS:
        if column not in frame:
            frame[column] = "unknown"
    frame["path"] = frame["path"].astype(str)
    missing_paths = ~frame["path"].map(lambda value: Path(value).exists())
    if missing_paths.any():
        raise FileNotFoundError(f"{dataset}: 오디오 경로 누락 {int(missing_paths.sum())}개")
    return frame[STANDARD_MANIFEST_COLUMNS]


def folder_manifest(
    root: Path, dataset: str, stage: str,
    bonafide_folders=("real", "bonafide", "genuine"),
    spoof_folders=("fake", "spoof", "synthetic"),
) -> pd.DataFrame:
    rows = []
    label_map = {name.lower(): 1 for name in bonafide_folders} | {name.lower(): 0 for name in spoof_folders}
    for path in root.rglob("*"):
        if not path.is_file() or path.suffix.lower() not in AUDIO_SUFFIXES:
            continue
        parents = {part.lower() for part in path.parts}
        matches = [label for name, label in label_map.items() if name in parents]
        if len(set(matches)) != 1:
            continue
        rows.append({"id": f"{dataset}:{path.stem}", "path": str(path), "label": matches[0]})
    if not rows:
        return pd.DataFrame(columns=STANDARD_MANIFEST_COLUMNS)
    return normalize_manifest(pd.DataFrame(rows), dataset, stage)


def build_robust_manifest(asv_frame: pd.DataFrame, specs: list[ExternalDatasetSpec]) -> pd.DataFrame:
    frames = []
    if len(asv_frame):
        built_in = asv_frame.rename(columns={"track": "content_type"}).copy()
        built_in["dataset"] = "ASVspoof2019_" + asv_frame["track"].astype(str)
        built_in["stage"] = built_in["split"].map({"train": "train", "dev": "selection", "eval": "audit"})
        built_in["generator"] = built_in["attack"].astype(str)
        built_in["codec"] = "original"
        frames.append(built_in[STANDARD_MANIFEST_COLUMNS])
    for spec in specs:
        path = Path(spec.manifest_path)
        if not path.exists():
            print(f"[skip] manifest 없음: {spec.name} -> {path}")
            continue
        external = normalize_manifest(pd.read_csv(path), spec.name, spec.stage)
        external["license_approved"] = spec.license_approved
        if spec.stage == "train" and not spec.license_approved:
            raise PermissionError(f"{spec.name}: 학습 사용 전 license_approved=True 확인 필요")
        frames.append(external[STANDARD_MANIFEST_COLUMNS])
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=STANDARD_MANIFEST_COLUMNS)


robust_manifest = build_robust_manifest(asv_df, EXTERNAL_DATASETS)
if len(robust_manifest):
    conflicting_labels = robust_manifest.groupby("path")["label"].nunique()
    if (conflicting_labels > 1).any():
        raise ValueError(f"같은 오디오에 상충 label 존재: {int((conflicting_labels > 1).sum())}개")
    stage_counts = robust_manifest.groupby("path")["stage"].nunique()
    if (stage_counts > 1).any():
        leaked = stage_counts[stage_counts > 1].index[:5].tolist()
        raise RuntimeError(f"train/selection/audit 간 동일 파일 누출: {leaked}")
    domain_labels = robust_manifest.groupby(["stage", "dataset"])["label"].nunique()
    invalid_domains = domain_labels[domain_labels < 2]
    if len(invalid_domains):
        raise ValueError(f"binary EER 계산이 불가능한 단일-class domain: {invalid_domains.to_dict()}")
    display(pd.crosstab([robust_manifest["stage"], robust_manifest["dataset"]], robust_manifest["label"], margins=True))


## B2. 고급 EDA: silence·bitrate·고주파·phase

Phase 통계는 강한 codec/resampling에도 변하므로 모델 입력으로 바로 사용하기보다 domain 차이를 진단하는 용도로 먼저 사용합니다. 한 데이터셋에서만 fake/real 차이가 나는 특징은 leakage 후보입니다.


In [ ]:
import librosa


def advanced_audio_metadata(path: str, max_seconds: float = 12.0) -> dict:
    info = sf.info(path)
    wav, sr = sf.read(path, dtype="float32", always_2d=True)
    wav = wav.mean(axis=1)[: int(info.samplerate * max_seconds)]
    if info.samplerate != CFG.sample_rate:
        wav = librosa.resample(wav, orig_sr=info.samplerate, target_sr=CFG.sample_rate)
        sr = CFG.sample_rate
    else:
        sr = info.samplerate
    duration = max(info.duration, 1e-6)
    bitrate_kbps = Path(path).stat().st_size * 8 / duration / 1000
    rms_frames = librosa.feature.rms(y=wav, frame_length=1024, hop_length=256)[0]
    reference = max(float(rms_frames.max()), 1e-8)
    silence_ratio = float(np.mean(20 * np.log10(np.maximum(rms_frames, 1e-8) / reference) < -40))

    stft = librosa.stft(wav, n_fft=1024, hop_length=256, win_length=1024)
    magnitude = np.abs(stft)
    frequencies = librosa.fft_frequencies(sr=sr, n_fft=1024)
    high_mask = frequencies >= min(6000, sr / 2 * 0.75)
    high_ratio = float(magnitude[high_mask].sum() / max(magnitude.sum(), 1e-8))
    phase = np.angle(stft)
    group_delay = -np.diff(np.unwrap(phase, axis=0), axis=0)
    instantaneous_frequency = np.diff(np.unwrap(phase, axis=1), axis=1)
    return {
        "duration": info.duration,
        "sample_rate": info.samplerate,
        "channels": info.channels,
        "format": info.format,
        "subtype": info.subtype,
        "bitrate_kbps_est": bitrate_kbps,
        "silence_ratio": silence_ratio,
        "high_frequency_ratio": high_ratio,
        "group_delay_std": float(np.nanstd(group_delay)),
        "instantaneous_frequency_std": float(np.nanstd(instantaneous_frequency)),
    }


def run_cross_domain_eda(manifest: pd.DataFrame, per_domain: int = 100) -> pd.DataFrame:
    if manifest.empty:
        return pd.DataFrame()
    # pandas 2.0~2.2에서 동일하게 동작하도록 groupby.apply의 버전별 인자를 피합니다.
    sampled_groups = [
        group.sample(min(per_domain, len(group)), random_state=CFG.seed)
        for _, group in manifest.groupby(["dataset", "label"], sort=False)
    ]
    sample = pd.concat(sampled_groups, ignore_index=True)
    rows = []
    for row in tqdm(sample.itertuples(index=False), total=len(sample), desc="Cross-domain EDA"):
        try:
            rows.append({"id": row.id, "dataset": row.dataset, "label": row.label, **advanced_audio_metadata(row.path)})
        except Exception as exc:
            print("EDA error:", row.path, repr(exc))
    result = pd.DataFrame(rows)
    if len(result):
        display(result.groupby(["dataset", "label"]).agg(
            n=("id", "size"), duration=("duration", "median"),
            silence=("silence_ratio", "mean"), high_freq=("high_frequency_ratio", "mean"),
            group_delay=("group_delay_std", "mean"), bitrate=("bitrate_kbps_est", "median"),
        ).round(4))
    return result


RUN_ADVANCED_EDA = False
advanced_eda_df = run_cross_domain_eda(robust_manifest) if RUN_ADVANCED_EDA else pd.DataFrame()


## B3. 실제 codec cache와 RIR augmentation

Codec은 단순 resampling이 아니라 FFmpeg encoder/decoder를 실제로 통과시킵니다. 학습 중 매 sample마다 실행하면 지나치게 느리므로 train 일부를 Drive/로컬 cache로 미리 생성합니다. AMR encoder는 Colab FFmpeg build에 없을 수 있어 실패 시 자동 제외합니다.


In [ ]:
import tempfile

CODEC_SETTINGS = {
    "mp3_64k": (".mp3", ["-codec:a", "libmp3lame", "-b:a", "64k"]),
    "aac_64k": (".m4a", ["-codec:a", "aac", "-b:a", "64k"]),
    "opus_32k": (".ogg", ["-codec:a", "libopus", "-b:a", "32k"]),
    "g711_mulaw": (".wav", ["-codec:a", "pcm_mulaw", "-ar", "8000"]),
    "g711_alaw": (".wav", ["-codec:a", "pcm_alaw", "-ar", "8000"]),
    "amr_nb": (".amr", ["-codec:a", "libopencore_amrnb", "-ar", "8000", "-ac", "1", "-b:a", "12.2k"]),
}


def ffmpeg_codec_roundtrip(source: Path, destination: Path, codec_name: str) -> bool:
    suffix, encode_args = CODEC_SETTINGS[codec_name]
    destination.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.TemporaryDirectory() as temporary:
        encoded = Path(temporary) / f"encoded{suffix}"
        encode = ["ffmpeg", "-hide_banner", "-loglevel", "error", "-y", "-i", str(source), "-ac", "1", *encode_args, str(encoded)]
        decode = ["ffmpeg", "-hide_banner", "-loglevel", "error", "-y", "-i", str(encoded), "-ac", "1", "-ar", str(CFG.sample_rate), str(destination)]
        try:
            subprocess.run(encode, check=True, timeout=90)
            subprocess.run(decode, check=True, timeout=90)
            return True
        except (subprocess.CalledProcessError, subprocess.TimeoutExpired) as exc:
            print(f"[codec skip] {codec_name}: {exc}")
            return False


def build_codec_cache(frame: pd.DataFrame, codecs=("mp3_64k", "aac_64k", "opus_32k", "g711_mulaw"), max_per_codec: int = 3000) -> pd.DataFrame:
    cache_root = DATA_ROOT / "codec_cache"
    rows = []
    base = frame.sample(min(max_per_codec, len(frame)), random_state=CFG.seed)
    for codec_name in codecs:
        for row in tqdm(base.itertuples(index=False), total=len(base), desc=f"codec:{codec_name}"):
            output = cache_root / codec_name / f"{str(row.id).replace(':', '_')}.wav"
            if output.exists() or ffmpeg_codec_roundtrip(Path(row.path), output, codec_name):
                item = row._asdict()
                item.update({
                    "id": f"{row.id}:{codec_name}", "path": str(output),
                    "dataset": f"{row.dataset}_codec", "codec": codec_name,
                })
                rows.append(item)
    return pd.DataFrame(rows)[STANDARD_MANIFEST_COLUMNS] if rows else pd.DataFrame(columns=STANDARD_MANIFEST_COLUMNS)


class RIRNoiseAugment:
    def __init__(self, probability: float = 0.25, noise_probability: float = 0.25):
        self.p = probability
        self.noise_p = noise_probability

    def __call__(self, wav: torch.Tensor) -> torch.Tensor:
        if random.random() < self.p:
            seconds = random.uniform(0.08, 0.45)
            length = max(64, int(seconds * CFG.sample_rate))
            time_axis = torch.arange(length, dtype=wav.dtype, device=wav.device) / CFG.sample_rate
            decay = torch.exp(-time_axis * random.uniform(8, 35))
            rir = decay * torch.randn_like(decay) * 0.08
            rir[0] += 1.0
            rir = rir / rir.abs().sum().clamp_min(1e-6)
            wav = F.conv1d(wav[None, None], rir.flip(0)[None, None], padding=length - 1)[0, 0, : wav.numel()]
        if random.random() < self.noise_p:
            snr_db = random.uniform(12, 35)
            signal = wav.square().mean().clamp_min(1e-8)
            wav = wav + torch.randn_like(wav) * (signal / (10 ** (snr_db / 10))).sqrt()
        return wav.clamp(-1, 1)


class ConservativeWaveAugment:
    """label 의미를 훼손하지 않는 작은 범위의 pitch/speed 변화만 적용합니다."""

    def __init__(self, pitch_probability: float = 0.0, speed_probability: float = 0.0):
        self.pitch_p = pitch_probability
        self.speed_p = speed_probability

    def __call__(self, wav: torch.Tensor) -> torch.Tensor:
        values = wav.detach().cpu().numpy().astype(np.float32, copy=False)
        if random.random() < self.pitch_p:
            values = librosa.effects.pitch_shift(
                values, sr=CFG.sample_rate, n_steps=random.uniform(-0.6, 0.6)
            ).astype(np.float32, copy=False)
        if random.random() < self.speed_p:
            values = librosa.effects.time_stretch(
                values, rate=random.uniform(0.95, 1.05)
            ).astype(np.float32, copy=False)
        return torch.from_numpy(np.ascontiguousarray(values)).clamp(-1, 1)


class RobustAudioFrameDataset(AudioFrameDataset):
    """Part A의 RawBoost/통신 증강에 RIR/noise와 보수적 pitch/speed를 연결한 학습 dataset."""

    def __init__(
        self, *args, rir_probability: float = 0.0, noise_probability: float = 0.0,
        pitch_probability: float = 0.0, speed_probability: float = 0.0, **kwargs,
    ):
        super().__init__(*args, **kwargs)
        self.rir_noise = RIRNoiseAugment(rir_probability, noise_probability)
        self.wave_augment = ConservativeWaveAugment(pitch_probability, speed_probability)

    def __getitem__(self, index: int) -> dict:
        row = self.frame.iloc[index]
        wav = load_mono(row["path"], CFG.sample_rate)
        if self.train and self.rawboost is not None:
            wav = self.rawboost(wav)
        if self.train:
            wav = self.communication(wav)
            wav = self.rir_noise(wav)
            wav = self.wave_augment(wav)
        wav = fixed_length(wav, self.clip_samples, self.train)
        labels = row[self.label_cols].to_numpy(dtype=np.float32)
        label = (
            torch.tensor(int(labels[0]), dtype=torch.long)
            if len(self.label_cols) == 1 else torch.tensor(labels, dtype=torch.float32)
        )
        return {"audio": wav, "label": label, "id": str(row["id"])}


BUILD_CODEC_CACHE = False
if BUILD_CODEC_CACHE:
    source = robust_manifest[robust_manifest["stage"] == "train"]
    codec_manifest = build_codec_cache(source)
    codec_manifest.to_csv(DRIVE_ROOT / "manifests/generated_codec_train.csv", index=False)
    robust_manifest = pd.concat([robust_manifest, codec_manifest], ignore_index=True)
    print("codec cache를 train manifest에 추가:", len(codec_manifest))


## B4. LFCC + Multi-scale Log-Mel ConvNeXt-style 모델

LFCC는 고주파를 Mel보다 덜 압축합니다. Multi-scale Log-Mel은 짧은 vocoder artifact와 긴 음악 구조를 동시에 봅니다. SpecAugment는 이 spectrogram branch의 train mode에서만 적용됩니다.


In [ ]:
class ConvNeXt2DBlock(nn.Module):
    def __init__(self, dim: int, drop: float = 0.0):
        super().__init__()
        self.depthwise = nn.Conv2d(dim, dim, 7, padding=3, groups=dim)
        self.norm = nn.GroupNorm(1, dim)
        self.pointwise = nn.Sequential(
            nn.Conv2d(dim, dim * 4, 1), nn.GELU(), nn.Dropout(drop), nn.Conv2d(dim * 4, dim, 1)
        )

    def forward(self, x):
        return x + self.pointwise(self.norm(self.depthwise(x)))


class SpectralEncoder(nn.Module):
    def __init__(self, width: int = 48, dropout: float = 0.15):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, width, 4, stride=4), ConvNeXt2DBlock(width, dropout),
            nn.Conv2d(width, width * 2, 2, stride=2), ConvNeXt2DBlock(width * 2, dropout),
            nn.Conv2d(width * 2, width * 4, 2, stride=2), ConvNeXt2DBlock(width * 4, dropout),
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
        )
        self.output_dim = width * 4

    def forward(self, x):
        return self.net(x)


class MultiScaleMelLFCC(nn.Module):
    def __init__(self, num_outputs: int = 2, dropout: float = 0.25):
        super().__init__()
        settings = [(512, 160, 80), (1024, 256, 96), (2048, 512, 128)]
        self.mels = nn.ModuleList([
            torchaudio.transforms.MelSpectrogram(
                sample_rate=CFG.sample_rate, n_fft=n_fft, win_length=n_fft,
                hop_length=hop, n_mels=n_mels, f_min=20, f_max=7600,
            ) for n_fft, hop, n_mels in settings
        ])
        self.lfcc = torchaudio.transforms.LFCC(
            sample_rate=CFG.sample_rate, n_filter=128, n_lfcc=60,
            speckwargs={"n_fft": 1024, "win_length": 1024, "hop_length": 256},
        )
        self.encoders = nn.ModuleList([SpectralEncoder() for _ in range(4)])
        self.freq_mask = torchaudio.transforms.FrequencyMasking(freq_mask_param=10)
        self.time_mask = torchaudio.transforms.TimeMasking(time_mask_param=18)
        dim = sum(encoder.output_dim for encoder in self.encoders)
        self.head = nn.Sequential(nn.LayerNorm(dim), nn.Dropout(dropout), nn.Linear(dim, 256), nn.GELU(), nn.Dropout(dropout), nn.Linear(256, num_outputs))

    def normalize_feature(self, feature: torch.Tensor) -> torch.Tensor:
        feature = (feature - feature.mean((-2, -1), keepdim=True)) / (feature.std((-2, -1), keepdim=True) + 1e-5)
        if self.training:
            feature = self.time_mask(self.freq_mask(feature))
        return feature.unsqueeze(1)

    def forward(self, audio: torch.Tensor) -> torch.Tensor:
        features = [torch.log(transform(audio).clamp_min(1e-6)) for transform in self.mels]
        features.append(self.lfcc(audio))
        embeddings = [encoder(self.normalize_feature(feature)) for encoder, feature in zip(self.encoders, features)]
        return self.head(torch.cat(embeddings, dim=-1))


## B5. WavLM/XLS-R + Temporal-Channel Modeling

비슷한 SSL encoder를 모두 학습하기보다 XLS-R와 WavLM 중 교차 데이터셋 성능이 좋은 하나를 주 모델로 선택합니다. TCM은 시간 convolution, channel gate, self-attention을 결합합니다.


In [ ]:
class TemporalChannelBlock(nn.Module):
    def __init__(self, dim: int, heads: int = 4, dropout: float = 0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.temporal = nn.Conv1d(dim, dim, kernel_size=7, padding=3, groups=dim)
        self.channel_gate = nn.Sequential(
            nn.Linear(dim, max(16, dim // 8)), nn.SiLU(), nn.Linear(max(16, dim // 8), dim), nn.Sigmoid()
        )
        self.norm2 = nn.LayerNorm(dim)
        self.attention = nn.MultiheadAttention(dim, heads, dropout=dropout, batch_first=True)
        self.ff = nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, dim * 4), nn.GELU(), nn.Dropout(dropout), nn.Linear(dim * 4, dim))

    def forward(self, x):
        residual = x
        z = self.norm1(x)
        temporal = self.temporal(z.transpose(1, 2)).transpose(1, 2)
        gate = self.channel_gate(z.mean(1)).unsqueeze(1)
        x = residual + temporal * gate
        z = self.norm2(x)
        x = x + self.attention(z, z, z, need_weights=False)[0]
        return x + self.ff(x)


class SSLTCMClassifier(SSLBackbone):
    def __init__(self, model_name: str, freeze: bool = True, dim: int = 256, depth: int = 3, dropout: float = 0.2, num_outputs: int = 2):
        super().__init__(model_name, freeze)
        self.proj = nn.Linear(self.hidden_size, dim)
        self.blocks = nn.Sequential(*[TemporalChannelBlock(dim, dropout=dropout) for _ in range(depth)])
        self.pool = AttentivePool(dim)
        self.head = nn.Sequential(nn.LayerNorm(dim * 2), nn.Dropout(dropout), nn.Linear(dim * 2, num_outputs))

    def forward(self, audio):
        return self.head(self.pool(self.blocks(self.proj(self.forward_features(audio)))))


ROBUST_EXPERIMENTS = {
    "exp101_multiscale_mel_lfcc": dict(
        model="multiscale_lfcc", batch_size=8, eval_batch_size=16, grad_accum=2,
        epochs=20, lr=2e-4, backbone_lr=2e-4, weight_decay=1e-4,
        clip_samples=96_000, patience=6, dropout=0.25,
        rir_probability=0.30, noise_probability=0.30,
        pitch_probability=0.10, speed_probability=0.10,
    ),
    "exp102_wavlm_tcm": dict(
        model="ssl_tcm", ssl_name="microsoft/wavlm-base-plus", batch_size=2, eval_batch_size=4,
        grad_accum=8, epochs=12, lr=1e-4, backbone_lr=8e-7, weight_decay=1e-4,
        clip_samples=96_000, freeze=True, freeze_epochs=2, unfreeze_last_n=4,
        patience=5, dropout=0.2, rir_probability=0.20, noise_probability=0.20,
        pitch_probability=0.05, speed_probability=0.05,
    ),
    "exp103_xlsr_tcm_rawboost": dict(
        model="ssl_tcm", ssl_name="facebook/wav2vec2-xls-r-300m", batch_size=2, eval_batch_size=4,
        grad_accum=8, epochs=14, lr=8e-5, backbone_lr=5e-7, weight_decay=1e-4,
        clip_samples=96_000, freeze=True, freeze_epochs=3, unfreeze_last_n=4,
        rawboost_probability=0.35, communication_probability=0.25,
        rir_probability=0.25, noise_probability=0.25,
        pitch_probability=0.05, speed_probability=0.05,
        patience=5, dropout=0.2,
    ),
}


def build_robust_model(exp: dict) -> nn.Module:
    if exp["model"] == "multiscale_lfcc":
        return MultiScaleMelLFCC(dropout=exp.get("dropout", 0.25))
    if exp["model"] == "ssl_tcm":
        return SSLTCMClassifier(exp["ssl_name"], freeze=exp.get("freeze", True), dropout=exp.get("dropout", 0.2))
    return build_model(exp)


## B6. Group-aware OOF와 강건성 checkpoint 기준

동일 화자·원본·생성기가 fold를 넘지 않도록 group을 구성합니다. 공식 ASVspoof split은 그대로 유지하고, 공개 DACON train이나 외부 학습 데이터에만 `StratifiedGroupKFold`를 적용합니다.

Checkpoint의 accuracy 강건성 점수는 높을수록 좋습니다.

`robust_accuracy = 0.5 × mean_accuracy + 0.5 × worst_accuracy - 0.1 × domain_std`

여기서 `worst_accuracy`는 selection domain 중 가장 낮은 accuracy입니다. 평균만 높은 모델보다 최악 domain에서도 버티는 모델을 우선합니다.

특정 domain 하나에서만 좋은 모델보다 모든 selection domain에서 고르게 작동하는 모델을 선택합니다.


In [ ]:
def assign_group_folds(frame: pd.DataFrame, n_splits: int = 3) -> pd.DataFrame:
    frame = frame.copy()
    speaker = frame["speaker"].fillna("unknown").astype(str)
    # speaker가 없을 때 전체가 하나의 group이 되는 오류를 피합니다. 가능하면 manifest에 speaker를 채우세요.
    identity = speaker.where(speaker.ne("unknown"), frame["id"].astype(str))
    groups = frame["dataset"].astype(str) + "|" + identity
    splitter = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=CFG.seed)
    frame["fold"] = -1
    for fold, (_, valid_index) in enumerate(splitter.split(frame, frame["label"], groups)):
        frame.loc[frame.index[valid_index], "fold"] = fold
    if (frame["fold"] < 0).any():
        raise RuntimeError("fold 할당 실패")
    return frame


def make_domain_loaders(manifest: pd.DataFrame, exp: dict, stage: str) -> dict[str, DataLoader]:
    loaders = {}
    for domain, frame in manifest[manifest["stage"] == stage].groupby("dataset"):
        dataset = AudioFrameDataset(frame.reset_index(drop=True), train=False, clip_samples=exp["clip_samples"])
        loaders[domain] = DataLoader(
            dataset, batch_size=exp.get("eval_batch_size", exp["batch_size"]),
            shuffle=False, num_workers=CFG.num_workers, pin_memory=True,
        )
    return loaders


@torch.no_grad()
def evaluate_domains(
    model: nn.Module, domain_loaders: dict[str, DataLoader],
    decision_threshold: float = 0.5, optimize_global_threshold: bool = False,
) -> tuple[pd.DataFrame, dict[str, pd.DataFrame]]:
    criterion = nn.CrossEntropyLoss()
    records, predictions = [], {}
    for domain, loader in domain_loaders.items():
        metrics, labels, scores, identifiers = run_epoch(model, loader, criterion)
        records.append({"domain": domain, **metrics})
        predictions[domain] = pd.DataFrame({"id": identifiers, "label": labels, "score_bonafide": scores})

    if optimize_global_threshold and predictions:
        pooled = pd.concat(predictions.values(), ignore_index=True)
        decision_threshold, _ = optimize_accuracy_threshold(
            pooled["label"].to_numpy(), pooled["score_bonafide"].to_numpy()
        )

    for record in records:
        domain_prediction = predictions[record["domain"]]
        labels = domain_prediction["label"].to_numpy()
        predicted = (domain_prediction["score_bonafide"].to_numpy() >= decision_threshold).astype(int)
        record["accuracy_at_0p5"] = record["accuracy"]
        record["accuracy"] = accuracy_score(labels, predicted)
        record["balanced_accuracy"] = balanced_accuracy_score(labels, predicted)
        record["f1"] = f1_score(labels, predicted, zero_division=0)

    report = pd.DataFrame(records)
    if len(report):
        mean_accuracy = report["accuracy"].mean()
        worst_accuracy = report["accuracy"].min()
        std_accuracy = report["accuracy"].std(ddof=0)
        robust_accuracy_score = 0.5 * mean_accuracy + 0.5 * worst_accuracy - 0.1 * std_accuracy
        report.attrs.update({
            "decision_threshold": float(decision_threshold),
            "mean_accuracy": mean_accuracy, "worst_accuracy": worst_accuracy,
            "std_accuracy": std_accuracy, "robust_accuracy_score": robust_accuracy_score,
        })
    return report, predictions


def plot_domain_report(report: pd.DataFrame, title: str = "Cross-dataset Accuracy"):
    if report.empty:
        return
    ordered = report.sort_values("accuracy", ascending=False)
    plt.figure(figsize=(9, max(3, 0.45 * len(ordered))))
    sns.barplot(data=ordered, x="accuracy", y="domain", color="#4C78A8")
    plt.axvline(ordered["accuracy"].mean(), color="#F58518", linestyle="--", label="mean accuracy")
    plt.title(title)
    plt.xlim(0, 1)
    plt.legend()
    plt.tight_layout()
    plt.show()


def manifest_sha256(frame: pd.DataFrame) -> str:
    stable = frame.sort_values(["dataset", "id"])[["id", "path", "label", "dataset", "stage"]]
    return hashlib.sha256(pd.util.hash_pandas_object(stable, index=False).values.tobytes()).hexdigest()


def save_robust_checkpoint(model, optimizer, exp_name, exp, epoch, selection_report, train_manifest, best=False):
    run_dir = RUN_ROOT / exp_name
    checkpoint_dir = run_dir / "checkpoints"
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    payload = {
        "model_state": model.state_dict(), "optimizer_state": optimizer.state_dict(),
        "epoch": epoch, "experiment": exp_name, "config": exp,
        "selection_metrics": selection_report.to_dict(orient="records"),
        "robust_accuracy_score": selection_report.attrs["robust_accuracy_score"],
        "decision_threshold": selection_report.attrs["decision_threshold"],
        "manifest_sha256": manifest_sha256(train_manifest),
        "repo_commits": REPO_COMMITS,
        "label_convention": {"0": "spoof", "1": "bonafide"},
    }
    epoch_path = checkpoint_dir / f"epoch_{epoch:03d}.pt"
    torch.save(payload, epoch_path)
    if best:
        shutil.copy2(epoch_path, run_dir / "best_robust.pt")
        (run_dir / "model_card.json").write_text(json.dumps({
            "experiment": exp_name, "robust_accuracy_score": payload["robust_accuracy_score"],
            "decision_threshold": payload["decision_threshold"],
            "selection_metrics": payload["selection_metrics"],
            "manifest_sha256": payload["manifest_sha256"],
            "sources": [asdict(spec) for spec in EXTERNAL_DATASETS],
        }, ensure_ascii=False, indent=2), encoding="utf-8")
    return epoch_path


## B7. Robust training과 untouched audit

아래 training은 selection domain의 robust accuracy score로 checkpoint를 고릅니다. Audit은 `RUN_FINAL_AUDIT=True`로 별도 실행하며 결과를 본 뒤 모델이나 threshold를 다시 바꾸면 audit이 더 이상 untouched가 아니므로 새 holdout이 필요합니다.


In [ ]:
from transformers import get_cosine_schedule_with_warmup


def train_one_epoch_with_warmup(model, loader, criterion, optimizer, scheduler, scaler, grad_accum):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    total = 0.0
    for step, batch in enumerate(tqdm(loader, desc="robust train", leave=False), 1):
        audio = batch["audio"].to(DEVICE, non_blocking=True)
        labels = batch["label"].to(DEVICE, non_blocking=True)
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=DEVICE.type == "cuda"):
            loss = criterion(model(audio), labels) / grad_accum
        scaler.scale(loss).backward()
        if step % grad_accum == 0 or step == len(loader):
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()
        total += float(loss.detach().cpu()) * grad_accum
    return total / max(1, len(loader))


def fit_robust(model, train_loader, selection_loaders, train_manifest, exp_name: str, exp: dict):
    model.to(DEVICE)
    optimizer = make_optimizer(model, exp)
    updates_per_epoch = math.ceil(len(train_loader) / exp.get("grad_accum", 1))
    total_updates = updates_per_epoch * exp["epochs"]
    scheduler = get_cosine_schedule_with_warmup(
        optimizer, num_warmup_steps=max(10, int(total_updates * 0.08)), num_training_steps=total_updates,
    )
    scaler = torch.cuda.amp.GradScaler(enabled=DEVICE.type == "cuda")
    criterion = nn.CrossEntropyLoss()
    best_score, stale, history = -float("inf"), 0, []
    for epoch in range(1, exp["epochs"] + 1):
        if epoch == exp.get("freeze_epochs", -1) + 1 and hasattr(model, "unfreeze_last_n"):
            model.unfreeze_last_n(exp.get("unfreeze_last_n", 4))
        train_loss = train_one_epoch_with_warmup(
            model, train_loader, criterion, optimizer, scheduler, scaler, exp.get("grad_accum", 1)
        )
        report, domain_predictions = evaluate_domains(
            model, selection_loaders, optimize_global_threshold=True
        )
        score = report.attrs["robust_accuracy_score"]
        improved = score > best_score
        if improved:
            best_score, stale = score, 0
        else:
            stale += 1
        save_robust_checkpoint(model, optimizer, exp_name, exp, epoch, report, train_manifest, best=improved)
        for domain, prediction in domain_predictions.items():
            path = RUN_ROOT / exp_name / "selection_predictions" / f"epoch_{epoch:03d}_{domain}.csv"
            path.parent.mkdir(parents=True, exist_ok=True)
            prediction.to_csv(path, index=False)
        record = {"epoch": epoch, "train_loss": train_loss, **report.attrs}
        history.append(record)
        pd.DataFrame(history).to_csv(RUN_ROOT / exp_name / "robust_history.csv", index=False)
        display(report)
        print(record)
        if stale >= exp["patience"]:
            break
    return pd.DataFrame(history)


SELECTED_ROBUST_EXPERIMENT = "exp101_multiscale_mel_lfcc"
RUN_ROBUST_TRAINING = False

if RUN_ROBUST_TRAINING:
    robust_exp = dict(ROBUST_EXPERIMENTS[SELECTED_ROBUST_EXPERIMENT])
    train_manifest = robust_manifest[robust_manifest["stage"] == "train"].reset_index(drop=True)
    if train_manifest.empty:
        raise RuntimeError("train stage manifest가 없습니다.")
    train_dataset = RobustAudioFrameDataset(
        train_manifest, train=True, clip_samples=robust_exp["clip_samples"],
        rawboost_probability=robust_exp.get("rawboost_probability", 0.0),
        communication_probability=robust_exp.get("communication_probability", 0.0),
        rir_probability=robust_exp.get("rir_probability", 0.0),
        noise_probability=robust_exp.get("noise_probability", 0.0),
        pitch_probability=robust_exp.get("pitch_probability", 0.0),
        speed_probability=robust_exp.get("speed_probability", 0.0),
    )
    train_loader = DataLoader(
        train_dataset, batch_size=robust_exp["batch_size"],
        sampler=balanced_sampler(train_manifest), num_workers=CFG.num_workers,
        pin_memory=True, drop_last=True,
    )
    selection_loaders = make_domain_loaders(robust_manifest, robust_exp, "selection")
    if len(selection_loaders) < 2:
        raise RuntimeError("교차 데이터셋 선택을 위해 selection domain이 최소 2개 필요합니다.")
    robust_model = build_robust_model(robust_exp)
    robust_history = fit_robust(
        robust_model, train_loader, selection_loaders, train_manifest,
        SELECTED_ROBUST_EXPERIMENT, robust_exp,
    )


In [ ]:
RUN_FINAL_AUDIT = False

if RUN_FINAL_AUDIT:
    checkpoint_path = RUN_ROOT / SELECTED_ROBUST_EXPERIMENT / "best_robust.pt"
    checkpoint = torch.load(checkpoint_path, map_location="cpu")
    robust_exp = checkpoint["config"]
    robust_model = build_robust_model(robust_exp)
    robust_model.load_state_dict(checkpoint["model_state"], strict=True)
    robust_model.to(DEVICE).eval()
    audit_loaders = make_domain_loaders(robust_manifest, robust_exp, "audit")
    audit_report, audit_predictions = evaluate_domains(
        robust_model, audit_loaders, decision_threshold=float(checkpoint["decision_threshold"]),
        optimize_global_threshold=False,
    )
    audit_dir = RUN_ROOT / SELECTED_ROBUST_EXPERIMENT / "final_audit"
    audit_dir.mkdir(parents=True, exist_ok=True)
    audit_report.to_csv(audit_dir / "cross_dataset_report.csv", index=False)
    for domain, prediction in audit_predictions.items():
        prediction.to_csv(audit_dir / f"{domain}_predictions.csv", index=False)
    summary = {**audit_report.attrs, "checkpoint": str(checkpoint_path)}
    (audit_dir / "summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
    display(audit_report.sort_values("accuracy", ascending=False))
    plot_domain_report(audit_report, "Untouched audit: domain Accuracy")
    print("FINAL AUDIT:", summary)


## B8. 권장 최종 조합

모든 후보를 넣는 것이 아니라 artifact 관점이 다른 모델만 남깁니다.

1. `AASIST + RawBoost/codec`: raw waveform의 국소 artifact
2. `WavLM 또는 XLS-R + TCM`: 언어·화자·통신환경의 context 일반화
3. `Multi-scale Log-Mel + LFCC`: 고주파·다중 시간축 artifact
4. 대회 label에 음악이 있으면 `speech/music multi-branch`

Selection OOF에서 상관관계가 높은 모델은 제거하고, 남은 모델의 constrained weight를 학습합니다. Audit 결과는 가중치 학습에 사용하지 않습니다.

| 사용자 기술 스택 | 이 통합안의 결정 |
|---|---|
| LFCC, Mel/STFT, multi-scale | 최종 spectral branch에 포함 |
| GD/IF, bitrate, silence | domain leakage를 찾는 EDA에 포함 |
| CQCC | LFCC와 중복되는 ablation 후보; 기본 ensemble에서는 제외 |
| codec, RIR, noise, SpecAugment, pitch/speed | train-only 증강으로 포함; 실제 codec은 cache 방식 |
| AASIST, WavLM/XLS-R, ConvNeXt 계열 | 관점이 다른 3개 핵심 branch로 포함 |
| HuBERT, RawNet3, Swin/EfficientNet, BigVGAN discriminator | 핵심 branch와 중복·비용이 커 기본 실행에서는 제외; selection EER 개선 시만 추가 |
| Multi-task | DACON 공개 target이 음성/음악을 분리할 때 Part A multi-branch에서 활성화 |
| AM-Softmax | 확률 calibration을 악화시킬 수 있어 기본 CE/BCE 유지; selection ablation에서만 채택 |
| StratifiedGroupKFold, cosine warmup, AMP | 포함 |

즉, 기술 수를 최대화한 조합이 아니라 **입력 관점의 다양성 대비 추론 비용**이 좋은 조합입니다. 제외 항목도 같은 selection protocol에서 이겼을 때만 편입합니다.


## B9. 첨부 이미지와 동일한 DACON 제출 ZIP

첨부 화면은 학습 checkpoint 형식이 아니라 **제출 ZIP의 최종 구조**입니다. 참가자가 ZIP에 넣는 것은 아래 세 항목뿐입니다.

```text
submit.zip
├── model/
│   ├── model_00.pt
│   ├── model_01.pt
│   └── ensemble.json
├── script.py
└── requirements.txt
```

`data/`와 `output/`은 평가 서버가 자동으로 추가합니다. 참가자 ZIP에 미리 포함하면 안 됩니다. `script.py`는 읽기 전용 `data/`를 파일 단위로 독립 추론하고 `output/submission.csv`만 생성합니다.


In [ ]:
ENSEMBLE_SUBMISSION_SCRIPT = r'''
from pathlib import Path
import json
import math
import numpy as np
import pandas as pd
import soundfile as sf
import torch
from scipy.signal import resample_poly

BASE = Path(__file__).resolve().parent
DATA = BASE / "data" if (BASE / "data").exists() else Path("/data")
OUTPUT = BASE / "output"
OUTPUT.mkdir(parents=True, exist_ok=True)
META = json.loads((BASE / "model" / "ensemble.json").read_text(encoding="utf-8"))
MODELS = [torch.jit.load(str(BASE / "model" / item["file"]), map_location="cpu").eval() for item in META["models"]]
SUFFIXES = {".wav", ".flac", ".ogg", ".mp3", ".m4a", ".aac"}

def load_mono(path):
    audio, sr = sf.read(path, dtype="float32", always_2d=True)
    audio = audio.mean(axis=1)
    target_sr = META["sample_rate"]
    if sr != target_sr:
        divisor = math.gcd(sr, target_sr)
        audio = resample_poly(audio, target_sr // divisor, sr // divisor).astype("float32")
    if len(audio) == 0:
        raise ValueError(f"empty audio: {path}")
    return torch.from_numpy(audio)

def make_segments(audio, length, count):
    if len(audio) <= length:
        return audio.repeat(math.ceil(length / len(audio)))[:length].unsqueeze(0)
    starts = np.linspace(0, len(audio) - length, count, dtype=int)
    return torch.stack([audio[start:start + length] for start in starts])

sample_paths = sorted(DATA.rglob("*sample*submission*.csv"))
if len(sample_paths) != 1:
    raise FileNotFoundError(f"sample_submission count={len(sample_paths)}")
sample = pd.read_csv(sample_paths[0])
id_col = META["id_col"]
targets = META["target_cols"]
audio_map = {path.stem: path for path in DATA.rglob("*") if path.is_file() and path.suffix.lower() in SUFFIXES}
rows = []
with torch.inference_mode():
    for identifier in sample[id_col].astype(str):
        key = Path(identifier).stem
        if key not in audio_map:
            raise FileNotFoundError(f"audio not found: {identifier}")
        audio = load_mono(audio_map[key])
        combined = None
        for model, item in zip(MODELS, META["models"]):
            clips = make_segments(audio, item["clip_samples"], item["segments"])
            probability = model(clips).mean(0).numpy()
            combined = probability * item["weight"] if combined is None else combined + probability * item["weight"]
        rows.append(combined)
predictions = np.stack(rows)
if predictions.ndim == 1:
    predictions = predictions[:, None]
if predictions.shape[1] != len(targets):
    raise ValueError((predictions.shape, targets))
sample.loc[:, targets] = predictions
if sample[targets].isna().any().any() or not np.isfinite(sample[targets].to_numpy()).all():
    raise ValueError("invalid probability")
sample.to_csv(OUTPUT / "submission.csv", index=False, encoding="utf-8")
'''


def build_ensemble_submit_zip(
    models: list[nn.Module], weights: Sequence[float], model_exps: list[dict],
    schema: dict, destination: Path,
) -> Path:
    weights = np.asarray(weights, dtype=np.float64)
    if len(models) != len(weights) or len(models) != len(model_exps):
        raise ValueError("models/weights/model_exps 길이가 다릅니다.")
    if np.any(weights < 0) or not np.isclose(weights.sum(), 1.0, atol=1e-6):
        raise ValueError("ensemble weight는 음수가 아니고 합이 1이어야 합니다.")
    package = WORK_ROOT / "robust_submit_package"
    if package.exists():
        shutil.rmtree(package)
    model_dir = package / "model"
    model_dir.mkdir(parents=True)
    model_metadata = []
    for index, (base_model, weight, model_exp) in enumerate(zip(models, weights, model_exps)):
        filename = f"model_{index:02d}.pt"
        wrapper = SubmissionProbabilityWrapper(
            base_model.eval().cpu(), len(schema["target_cols"]), schema["single_target_positive"]
        )
        example = torch.zeros(1, model_exp["clip_samples"])
        traced = torch.jit.trace(wrapper, example, strict=False)
        traced.save(str(model_dir / filename))
        model_metadata.append({
            "file": filename, "weight": float(weight),
            "clip_samples": int(model_exp["clip_samples"]), "segments": 5,
        })
    metadata = {
        "sample_rate": CFG.sample_rate, "id_col": schema["id_col"],
        "target_cols": schema["target_cols"], "models": model_metadata,
        "label_convention": {"0": "spoof", "1": "bonafide"},
    }
    (model_dir / "ensemble.json").write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8")
    (package / "script.py").write_text(ENSEMBLE_SUBMISSION_SCRIPT, encoding="utf-8")
    (package / "requirements.txt").write_text("soundfile\nscipy\npandas\n", encoding="utf-8")

    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists():
        destination.unlink()
    shutil.make_archive(str(destination.with_suffix("")), "zip", package)
    with zipfile.ZipFile(destination) as archive:
        names = archive.namelist()
        top_level = {name.split("/")[0] for name in names}
        if top_level != {"model", "script.py", "requirements.txt"}:
            raise RuntimeError(f"잘못된 ZIP top-level: {top_level}")
        if any(name.startswith("data/") or name.startswith("output/") for name in names):
            raise RuntimeError("data/output은 평가 서버가 추가하므로 ZIP에 포함하면 안 됩니다.")
        if "model/ensemble.json" not in names:
            raise RuntimeError("ensemble metadata 누락")
    print("submit zip:", destination, "MB=", round(destination.stat().st_size / 1024**2, 2))
    return destination


def load_best_robust_model(exp_name: str) -> tuple[nn.Module, dict, dict]:
    checkpoint_path = RUN_ROOT / exp_name / "best_robust.pt"
    checkpoint = torch.load(checkpoint_path, map_location="cpu")
    model_exp = checkpoint["config"]
    model = build_robust_model(model_exp)
    model.load_state_dict(checkpoint["model_state"], strict=True)
    return model.eval(), model_exp, checkpoint


def smoke_test_submit_zip(zip_path: Path, local_data_dir: Path, timeout_seconds: int = 900) -> pd.DataFrame:
    # 평가 서버의 data/output 배치를 임시 폴더에서 재현하고 제출물을 실제로 실행합니다.
    zip_path = Path(zip_path).resolve()
    local_data_dir = Path(local_data_dir).resolve()
    if not zip_path.is_file():
        raise FileNotFoundError(zip_path)
    if not local_data_dir.is_dir():
        raise NotADirectoryError(local_data_dir)
    with tempfile.TemporaryDirectory(prefix="dacon_submit_smoke_") as temporary:
        sandbox = Path(temporary) / "submit"
        sandbox.mkdir(parents=True)
        with zipfile.ZipFile(zip_path) as archive:
            archive.extractall(sandbox)
        shutil.copytree(local_data_dir, sandbox / "data")
        environment = os.environ.copy()
        environment.update({
            "HF_HUB_OFFLINE": "1", "TRANSFORMERS_OFFLINE": "1",
            "TOKENIZERS_PARALLELISM": "false",
        })
        result = subprocess.run(
            [sys.executable, "script.py"], cwd=sandbox, env=environment,
            capture_output=True, text=True, timeout=timeout_seconds,
        )
        if result.returncode != 0:
            raise RuntimeError(
                f"submission smoke test 실패(code={result.returncode})\n"
                f"stdout:\n{result.stdout[-4000:]}\nstderr:\n{result.stderr[-4000:]}"
            )
        output_path = sandbox / "output" / "submission.csv"
        if not output_path.is_file():
            raise FileNotFoundError("script.py가 output/submission.csv를 생성하지 않았습니다.")
        output = pd.read_csv(output_path)
        sample_paths = sorted((sandbox / "data").rglob("*sample*submission*.csv"))
        if len(sample_paths) != 1:
            raise RuntimeError(f"sample_submission count={len(sample_paths)}")
        sample = pd.read_csv(sample_paths[0])
        if list(output.columns) != list(sample.columns) or len(output) != len(sample):
            raise ValueError("submission의 열 또는 행 수가 sample_submission과 다릅니다.")
        numeric = output.select_dtypes(include=[np.number])
        if numeric.empty or not np.isfinite(numeric.to_numpy()).all():
            raise ValueError("submission에 유효한 수치 예측이 없습니다.")
        print("offline smoke test 통과:", output.shape, list(output.columns))
        return output.head()


BUILD_ROBUST_SUBMIT_ZIP = False
if BUILD_ROBUST_SUBMIT_ZIP:
    schema = infer_schema(dacon_contract, DACON_SCHEMA)
    # final_models / final_exps / ensemble_weights는 selection OOF에서 확정한 값만 사용합니다.
    robust_submit_zip = build_ensemble_submit_zip(
        final_models, ensemble_weights, final_exps, schema,
        EXPORT_ROOT / "submit.zip",
    )

RUN_SUBMIT_SMOKE_TEST = False
if RUN_SUBMIT_SMOKE_TEST:
    # local_data_dir에는 평가 형식을 복제한 audio와 sample_submission.csv가 있어야 합니다.
    display(smoke_test_submit_zip(EXPORT_ROOT / "submit.zip", DATA_ROOT / "dacon_smoke_data"))


## B10. 최종 실행 순서

1. ASVspoof LA/PA로 Part A smoke test를 통과합니다.
2. 사용 허가가 확인된 외부 데이터 manifest만 등록합니다.
3. 고급 EDA에서 dataset 자체의 bitrate·silence·codec이 label을 누설하는지 확인합니다.
4. `AASIST`, `WavLM/XLS-R+TCM`, `Multi-scale Mel+LFCC`를 각각 학습합니다.
5. 최소 2개의 selection domain으로 robust checkpoint를 선택합니다.
6. Selection OOF 예측만으로 ensemble weight·calibration·threshold를 결정합니다.
7. 모든 설정을 고정하고 audit dataset을 한 번 평가합니다.
8. 최종 checkpoint를 TorchScript `model_XX.pt`로 변환해 `submit.zip`을 생성합니다.
9. 로컬 복제 `data/`에서 `script.py`를 오프라인 실행하고 `output/submission.csv`를 검증합니다.

이 파일의 최적화 목표는 `mean accuracy`, `worst-domain accuracy`, `domain std`이며 balanced accuracy·F1·EER을 안전 지표로 함께 봅니다. 대회 공개 후에는 공식 평가 지표가 최종 기준입니다.
